# DEKAE: Dynamic Edge-driven Topology for Episodic Few-Shot Learning
## Extension of FSAKE with Supervised Relational Recovery

**Paper identity**: *A topology-learning framework with explicit structural regularization and supervised relational recovery for episodic few-shot learning.*

| Section | Description |
|---------|-------------|
| 1 | Environment Setup & GPU Check |
| 2 | Data Pipeline (Local) |
| 3 | Dataset & Episodic Loader |
| 4 | Backbone (Conv4) |
| 5 | Static k-NN Graph Module (HGNN Baseline — A1 ablation) |
| 6 | Edge Incidence Matrix & Dynamic Edge Features |
| 7 | Adaptive Topology Reconstruction |
| 8 | Knowledge Filtering with Edge Awareness |
| 9 | Supervised Edge Loss (Support-Support Only) |
| 10 | Sparsity Regularization |
| 11 | Full Model Assembly (DEKAE) |
| 12 | Training Loop with Checkpointing |
| **12b** | **Execute Training ← RUN THIS to train on CIFAR-FS** |
| 13 | Evaluation & Statistical Testing (dummy sanity check) |
| **13b** | **Real Test-Set Evaluation ← RUN AFTER 12b** |
| 14 | Synthetic Graph Recovery Experiment |
| 15 | Ablation Study Runner |
| **15b** | **Execute Ablation Groups A + E ← RUN AFTER 12b** |
| 15c | Ablation Group C — Edge Feature Comparison |
| 15d | Ablation Group D — Seed Stability & Topology Init Sensitivity |
| 16 | Metrics Logging (W&B) |
| 17 | Visualization Suite |

> **Ablation A1 vs A2 vs DEKAE**: A1 (`A1_HGNN`) = static k-NN adjacency (HGNN-style). A2 (`A2_FSAKE`) = FSAKE's actual dynamic scalar-adjacency (MLP on node differences, no edge features). DEKAE (`A5+`) = vector-valued edge incidence + supervised edge-level loss. **FSAKE does NOT use static k-NN** — it recomputes `A^(l) = softmax(MLP(|h_i − h_j|))` at every layer. The static k-NN is the HGNN predecessor baseline.

---
**Execution order for a full run:**
1. Run cells 1–3 (setup, data, loaders)
2. Run cells 4–12 (architecture definitions)
3. ▶ **Run cell 12b** — starts real training (~60–90 min on T4)
4. ▶ **Run cell 13b** — evaluates on test set, prints accuracy table
5. ▶ **Run cell 14** — synthetic topology recovery (trained model)
6. ▶ **Run cell 15b** — ablation sweeps (Groups A & E)

## 🔒 Step 0 — Google Drive Mount (Run First)

Run the cell below **once per session** before any other cell.  
It mounts Google Drive and redirects all checkpoints, results, and cached dataset files there so they survive a runtime reset.

| Path on Drive | Purpose |
|---|---|
| `MyDrive/DEKAE_Project/checkpoints/` | Best + periodic epoch `.pth` files |
| `MyDrive/DEKAE_Project/results/` | JSON training history, ablation results |
| `MyDrive/DEKAE_Project/data/` | CIFAR-FS cached download (skip re-download) |

> If Drive is unavailable the notebook falls back to `/content/` (files lost on reset).


In [ ]:
# ── Step 0: Mount Google Drive and configure persistent paths ─────────────────
# Run this cell FIRST, before all others.
#
# Everything — checkpoints, training history, and the cached dataset archive —
# is saved to YOUR Google Drive account under:
#
#   My Drive / DEKAE_Project /
#       ├── checkpoints/     ← best model + periodic epoch .pth files
#       ├── results/         ← training history JSON
#       └── data/            ← dataset archive (re-used across sessions)
#
# No manual setup required. The folder is created automatically on first run.

import pathlib, os

# ── Project folder path inside the signed-in Drive account ───────────────────
_MYDRV_PATH = pathlib.Path("/content/drive/MyDrive/DEKAE_Project")

DRIVE_ROOT: pathlib.Path = None

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    _drive_base = _MYDRV_PATH

    # Create the project folder and sub-directories if they don't exist yet.
    for _sub in ("checkpoints", "results", "data"):
        _sub_path = _drive_base / _sub
        try:
            _sub_path.mkdir(parents=True, exist_ok=True)
        except OSError:
            pass  # already exists — harmless
        print(f"  ✓ {_sub}/  →  {_sub_path}" if _sub_path.exists()
              else f"  ⚠ '{_sub}/' not visible yet — re-run this cell once.")

    DRIVE_ROOT = _drive_base
    USE_DRIVE  = True
    print(f"\n✓ Google Drive mounted.")
    print(f"  Checkpoints → {DRIVE_ROOT / 'checkpoints'}")
    print(f"  Results     → {DRIVE_ROOT / 'results'}")
    print(f"  Data cache  → {DRIVE_ROOT / 'data'}")

except Exception as _e:
    # ── Local fallback (running outside Colab / Drive unavailable) ────────────
    print(f"⚠  Google Drive not available ({_e}).")
    print("   Falling back to local storage next to this notebook.")
    _NOTEBOOK_DIR = pathlib.Path(os.path.abspath(""))
    DRIVE_ROOT    = _NOTEBOOK_DIR / "DEKAE_Project"
    for _sub in ("checkpoints", "results", "data"):
        (DRIVE_ROOT / _sub).mkdir(parents=True, exist_ok=True)
        print(f"  ✓ {_sub}/  →  {DRIVE_ROOT / _sub}")
    USE_DRIVE = False
    print(f"\n✓ Local fallback root → {DRIVE_ROOT}")

print(f"\n   USE_DRIVE = {USE_DRIVE}")


Mounted at /content/drive
✓ Resolved DEKAE_Project via folder ID  → /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project
  Drive listing: ['checkpoints', 'results', 'data']
  ✓ checkpoints/
  ✓ results/
  ✓ data/
✓ Google Drive mounted.  Persistent root → /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project
   USE_DRIVE = True


In [2]:
# ── Section 1: Environment Setup & Dependencies ──────────────────────────────
# Run once per Colab session. Installs torch-geometric matching the Colab CUDA version.

import sys
import torch
import os

# 1. Downgrade setuptools to avoid build errors with older packages
!pip install -q "setuptools<70" wheel Cython

# 2. Install standard dependencies
!pip install -q torch-geometric wandb scipy scikit-learn matplotlib seaborn networkx tqdm Pillow

# 3. Install learn2learn from source in editable mode (bypasses wheel build failure)
if not os.path.exists("/content/learn2learn"):
    !git clone https://github.com/learnables/learn2learn.git /content/learn2learn

# Install with -e (editable) to skip bdist_wheel
!cd /content/learn2learn && pip install -e . -q

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("VRAM            :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠ No GPU detected — switch Runtime → Change runtime type → GPU")

# ── Reproducibility ───────────────────────────────────────────────────────────
import random, numpy as np

GLOBAL_SEED = 42

def set_seed(seed: int = GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.7 MB/s eta 0:00:00
Cloning into '/content/learn2learn'...
remote: Enumerating objects: 4881, done.
remote: Counting objects: 100% (1221/1221), done.
remote: Compressing objects: 100% (307/307), done.
remote: Total 4881 (delta 1041), reused 914 (delta 914), pack-reused 3660 (from 1)
Receiving objects: 100% (4881/4881), 9.46 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (2834/2834), done.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ..

## Section 2: Data Pipeline (Local)

In [ ]:
# ── Section 2: Data Download + Local Project Setup ───────────────────────────
#
# Dataset always lives on the local drive under DRIVE_ROOT/data/.
#
# Fast-resume strategy (no re-download on run 2+):
#   • First run   : download from internet → archive to DRIVE_ROOT/data/
#   • Later runs  : extract archive (~30 s) + index rebuild (~15 s, automatic)
#
# If you see a FileNotFoundError pointing to a /content/drive/... path:
#   The archive was built when data lived on Drive FUSE.
#   Fix: the code below deletes the stale archive automatically and re-downloads.

import shutil, tarfile, pathlib, sys, os, time

# ── Fix Import Shadowing ─────────────────────────────────────────────────────
if "learn2learn" in sys.modules:
    import learn2learn as _l2l_check
    if not hasattr(_l2l_check, "__file__") or _l2l_check.__file__ is None:
        print("Unloading broken learn2learn namespace package...")
        del sys.modules["learn2learn"]

if os.path.exists("/content/learn2learn"):
    if "/content/learn2learn" not in sys.path:
        sys.path.insert(0, "/content/learn2learn")

import learn2learn as l2l
print(f"learn2learn imported from: {getattr(l2l, '__file__', 'unknown')}")

# ─────────────────────────────────────────────────────────────────────────────
ACTIVE_DATASET = 'cifarFS'

# Dataset root — always on the local drive under DRIVE_ROOT
LOCAL_DATA_ROOT = DRIVE_ROOT / "data"
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ LOCAL_DATA_ROOT = {LOCAL_DATA_ROOT}  (local drive)")

# ── processed/ subdir that learn2learn uses for absolute-path pickle index ───
_DS_SUBDIR = {
    'cifarFS'       : 'cifarfs',
    'miniImageNet'  : 'mini-imagenet',
    'tieredImageNet': 'tiered-imagenet',
    'cub'           : 'cub200',
}
_ds_subdir = _DS_SUBDIR.get(ACTIVE_DATASET, ACTIVE_DATASET.lower())
_processed = LOCAL_DATA_ROOT / _ds_subdir / "processed"

def _purge_processed():
    """Delete the stale pickle index so learn2learn rebuilds it with local paths."""
    if _processed.exists():
        shutil.rmtree(str(_processed))
        print(f"  Deleted stale index: {_processed}  (will be rebuilt with local paths)")

# ── Step 0 (guard): purge any stale dataset directories from DRIVE_ROOT/data/ ─
_ARCHIVE_NAME  = f"{ACTIVE_DATASET}_data.tar.gz"
_drive_archive = (DRIVE_ROOT / "data" / _ARCHIVE_NAME) if USE_DRIVE else None

if USE_DRIVE:
    _drive_data_dir = DRIVE_ROOT / "data"
    try:
        for _item in list(_drive_data_dir.iterdir()):
            if _item.is_dir():
                print(f"  🗑  Removing stale dataset directory from Drive: {_item.name}/")
                print(f"     (Raw data on Drive FUSE causes learn2learn to embed Drive paths)")
                try:
                    shutil.rmtree(str(_item))
                except Exception as _re:
                    print(f"     Could not remove {_item.name}: {_re}")
    except Exception:
        pass  # drive_data_dir not listable yet — harmless

# ── Step 1: Validate existing archive (reject if it references Drive paths) ───
def _archive_is_clean() -> bool:
    """
    Peek inside the archive and check whether any member path contains the
    string 'drive' — a sign the archive was built when LOCAL_DATA_ROOT was
    on Drive FUSE.
    Returns True if the archive looks clean (safe to extract).
    """
    if _drive_archive is None or not _drive_archive.exists():
        return True
    try:
        with tarfile.open(str(_drive_archive), "r:gz") as tar:
            for m in tar.getmembers():
                if "drive" in m.name or m.name.startswith("/"):
                    return False
        return True
    except Exception:
        return False

if _drive_archive and _drive_archive.exists() and not _archive_is_clean():
    print(f"  🗑  Archive contains stale Drive-FUSE paths — deleting it.")
    print(f"     The dataset will be re-downloaded and a clean archive saved.")
    try:
        _drive_archive.unlink()
    except Exception as _ue:
        print(f"     Could not delete archive ({_ue}) — extraction will be skipped.")

# ── Step 2: Restore dataset from archive if available ────────────────────────
_marker = LOCAL_DATA_ROOT / f".{ACTIVE_DATASET}_extracted"

def _try_restore_from_archive() -> bool:
    """Extract the archive to the local drive. Returns True on success."""
    if _drive_archive is None or not _drive_archive.exists():
        return False
    if _marker.exists():
        print("  Dataset already extracted this session — skipping tar extraction.")
    else:
        print(f"  Extracting {_ARCHIVE_NAME} → {LOCAL_DATA_ROOT}  (~30 s) …")
        t0 = time.time()
        try:
            with tarfile.open(str(_drive_archive), "r:gz") as tar:
                tar.extractall(str(LOCAL_DATA_ROOT))
            _marker.touch()
            print(f"  Extraction complete in {time.time()-t0:.1f}s")
        except Exception as ex:
            print(f"  Archive extract failed ({ex}) — will download from internet.")
            return False
    _purge_processed()
    return True

_restored = _try_restore_from_archive()
if _restored:
    print(f"✓ Dataset restored from archive.")
    print(f"  Index (processed/) purged — learn2learn will rebuild it with local paths.")
else:
    print(f"  No valid archive found — downloading {ACTIVE_DATASET} from internet.")
    _purge_processed()

# ── Step 3: Load via learn2learn ──────────────────────────────────────────────
from learn2learn.vision.datasets import MiniImagenet, TieredImagenet, CIFARFS, CUBirds200
from torchvision import transforms

MEAN = [0.4712, 0.4499, 0.4031]
STD  = [0.2726, 0.2634, 0.2794]

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize(84),
    transforms.CenterCrop(84),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
AUG_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(84),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

DATASET_CLS = {
    'miniImageNet'  : MiniImagenet,
    'tieredImageNet': TieredImagenet,
    'cifarFS'       : CIFARFS,
    'cub'           : CUBirds200,
}

print(f"\nLoading {ACTIVE_DATASET} from {LOCAL_DATA_ROOT} …")
cls = DATASET_CLS[ACTIVE_DATASET]

try:
    train_dataset = cls(root=str(LOCAL_DATA_ROOT), mode='train',
                        download=True, transform=BASE_TRANSFORM)
    val_dataset   = cls(root=str(LOCAL_DATA_ROOT), mode='validation',
                        download=True, transform=BASE_TRANSFORM)
    test_dataset  = cls(root=str(LOCAL_DATA_ROOT), mode='test',
                        download=True, transform=BASE_TRANSFORM)

    train_dataset = l2l.data.MetaDataset(train_dataset)
    val_dataset   = l2l.data.MetaDataset(val_dataset)
    test_dataset  = l2l.data.MetaDataset(test_dataset)

    print(f"✓ {ACTIVE_DATASET} — "
          f"train: {len(train_dataset.labels_to_indices)} classes, "
          f"val:   {len(val_dataset.labels_to_indices)} classes, "
          f"test:  {len(test_dataset.labels_to_indices)} classes")
    _download_ok = True

except Exception as e:
    print(f"\n❌ Error loading {ACTIVE_DATASET}: {e}")
    if "/content/drive" in str(e):
        print("\n  Root cause: the archive was built while data lived on Drive FUSE.")
        print("  The code above should have caught and deleted it automatically.")
        print("  If this error persists, manually delete everything inside")
        print(f"  {DRIVE_ROOT}/data/ and re-run all cells.")
    raise e

# ── Step 4: Archive dataset after first download (one-time) ──────────────────
# Stores IMAGE FILES ONLY — processed/ excluded so index is always rebuilt locally.
if USE_DRIVE and _download_ok and not _restored:
    _drive_data_dir = DRIVE_ROOT / "data"
    try:
        _drive_data_dir.mkdir(parents=True, exist_ok=True)
    except OSError:
        pass

    if _drive_archive and _drive_archive.exists():
        try:
            _drive_archive.unlink()
        except Exception:
            pass

    print(f"\n  Archiving dataset for future sessions (one-time, ~2 min) …")
    print(f"  Excluding processed/ (rebuilt locally each session with correct paths)")
    t0 = time.time()
    try:
        with tarfile.open(str(_drive_archive), "w:gz") as tar:
            for entry in LOCAL_DATA_ROOT.rglob("*"):
                if "processed" in entry.parts:
                    continue
                if entry.is_file():
                    tar.add(str(entry), arcname=str(entry.relative_to(LOCAL_DATA_ROOT)))
        _marker.touch()
        sz_mb = _drive_archive.stat().st_size / 1e6
        print(f"  ✓ Archived in {time.time()-t0:.1f}s  ({sz_mb:.0f} MB) → {_drive_archive.name}")
    except Exception as ae:
        print(f"  ⚠ Archiving failed ({ae}). Next session will re-download from internet.")
elif _drive_archive and _drive_archive.exists():
    print(f"\n  Archive present: {_drive_archive.name}")

# ── Step 5: Persistent checkpoint / results dirs ─────────────────────────────
CKPT_DIR    = DRIVE_ROOT / "checkpoints"
RESULTS_DIR = DRIVE_ROOT / "results"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✓ Checkpoints → {CKPT_DIR}")
print(f"✓ Results     → {RESULTS_DIR}")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


learn2learn imported from: /content/learn2learn/learn2learn/__init__.py
✓ LOCAL_DATA_ROOT = /content/data  (local SSD)
  🗑  Removing stale dataset directory from Drive: cifarfs/
     (Raw data on Drive FUSE causes learn2learn to embed Drive paths)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  🗑  Drive archive contains stale Drive-FUSE paths — deleting it.
     The dataset will be re-downloaded from the internet this session
     and a clean archive will be saved to Drive for future sessions.
  No valid Drive archive found — downloading cifarFS from internet.

Loading cifarFS from /content/data …


Creating CIFARFS splits
✓ cifarFS — train: 64 classes, val:   16 classes, test:  20 classes

  Archiving dataset to Drive for future sessions (one-time, ~2 min) …
  Excluding processed/ (rebuilt locally each session with correct paths)
  ✓ Archived in 27.8s  (170 MB) → cifarFS_data.tar.gz

✓ Checkpoints → /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/checkpoints
✓ Results     → /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/results
learn2learn imported from: /content/learn2learn/learn2learn/__init__.py
✓ LOCAL_DATA_ROOT = /content/data  (local SSD)
  Dataset already extracted this session — skipping tar extraction.
  Deleted stale index: /content/data/cifarfs/processed  (will be rebuilt with local paths)
✓ Dataset image files restored from Drive archive.
  Index (processed/) purged — learn2learn will rebuild it with local paths.

Loading cifarFS from /content/data …
Creating CIFARFS splits
✓ cifarFS — trai

## Section 2: Google Drive Mount & Data Pipeline

In [ ]:
# ── Section 2: Data Download + Local Project Setup ───────────────────────────
# Strategy:
#   • DRIVE_ROOT / USE_DRIVE are set by the Step-0 Drive-mount cell above.
#   • When Drive is mounted, dataset, checkpoints and results all live on Drive
#     and survive runtime resets — you can resume training mid-run.
#   • When Drive is unavailable, everything falls back to /content/ (session-only).
#
# Supported datasets (all download automatically on first run):
#   ACTIVE_DATASET = 'miniImageNet'   ← default; required for fair comparison with FSAKE
#   Other options : 'tieredImageNet' | 'cifarFS' | 'cub'
#
# ⚠ Fair-comparison requirement (plan §2.2):
#   EVERYTHING must be identical to the FSAKE baseline — same dataset, splits,
#   backbone, feature dimension, episodic sampling, and optimizer — except the
#   graph construction and message passing.
#   FSAKE reports on miniImageNet → DEKAE must also use miniImageNet.
#   cifarFS or tieredImageNet may be used as ADDITIONAL benchmarks once
#   miniImageNet results are established.

import shutil, pathlib, zipfile, sys, os

# ── Fix Import Shadowing ─────────────────────────────────────────────────────
# The repo is at /content/learn2learn. The package is /content/learn2learn/learn2learn.
# If CWD is /content, 'import learn2learn' finds the repo folder (namespace pkg) first.

# 1. Force reload if already loaded incorrectly
if "learn2learn" in sys.modules:
    import learn2learn
    # If it lacks __file__, it's likely the namespace package (broken)
    if not hasattr(learn2learn, "__file__") or learn2learn.__file__ is None:
        print("Creating fix: unloading broken learn2learn module...")
        del sys.modules["learn2learn"]

# 2. Prepend repo path to sys.path so the inner package is found first
if os.path.exists("/content/learn2learn"):
    if "/content/learn2learn" not in sys.path:
        sys.path.insert(0, "/content/learn2learn")

import learn2learn as l2l
print(f"learn2learn imported from: {getattr(l2l, '__file__', 'unknown')}")

# ─────────────────────────────────────────────────────────────────────────────

# miniImageNet — same dataset as FSAKE baseline (required for fair comparison).
# If the download fails (Colab network instability), switch temporarily to
# 'cifarFS' for development, then revert before generating paper results.
ACTIVE_DATASET = 'miniImageNet'

# ── Data root: prefer Drive so the download is cached across sessions ─────────
if USE_DRIVE:
    LOCAL_DATA_ROOT = DRIVE_ROOT / "data"
    print(f"✓ Dataset cache → Google Drive: {LOCAL_DATA_ROOT}")
else:
    LOCAL_DATA_ROOT = DRIVE_ROOT / "data"
    print(f"✓ Dataset cache → local drive: {LOCAL_DATA_ROOT}")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── Step 1: Download dataset via learn2learn ─────────────────────────────────
from learn2learn.vision.datasets import (
    MiniImagenet, TieredImagenet, CIFARFS, CUBirds200
)

from torchvision import transforms

MEAN = [0.4712, 0.4499, 0.4031]
STD  = [0.2726, 0.2634, 0.2794]

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize(84),
    transforms.CenterCrop(84),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
AUG_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(84),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

DATASET_CLS = {
    'miniImageNet' : MiniImagenet,
    'tieredImageNet': TieredImagenet,
    'cifarFS'      : CIFARFS,
    'cub'          : CUBirds200,
}

print(f"Downloading / loading {ACTIVE_DATASET} …")
cls = DATASET_CLS[ACTIVE_DATASET]

# download=True fetches the data automatically on first run.
# Subsequent runs reuse the cached copy in LOCAL_DATA_ROOT.
try:
    train_dataset = cls(root=str(LOCAL_DATA_ROOT), mode='train',
                        download=True, transform=BASE_TRANSFORM)
    val_dataset   = cls(root=str(LOCAL_DATA_ROOT), mode='validation',
                        download=True, transform=BASE_TRANSFORM)
    test_dataset  = cls(root=str(LOCAL_DATA_ROOT), mode='test',
                        download=True, transform=BASE_TRANSFORM)

    # Wrap in MetaDataset to ensure labels_to_indices exists (needed for EpisodicSampler)
    train_dataset = l2l.data.MetaDataset(train_dataset)
    val_dataset   = l2l.data.MetaDataset(val_dataset)
    test_dataset  = l2l.data.MetaDataset(test_dataset)

    print(f"{ACTIVE_DATASET} — "
          f"train: {len(train_dataset.labels_to_indices)} classes, "
          f"val: {len(val_dataset.labels_to_indices)} classes, "
          f"test: {len(test_dataset.labels_to_indices)} classes")
    print(f"Data stored at : {LOCAL_DATA_ROOT}")

except Exception as e:
    print(f"\n❌ Error downloading {ACTIVE_DATASET}: {e}")
    print("Tip: If miniImageNet fails due to Colab network issues, temporarily switch")
    print("     ACTIVE_DATASET to 'cifarFS' for development — but revert before")
    print("     generating paper results (fair comparison with FSAKE requires miniImageNet).")
    raise e

# ── Step 2: Checkpoint / results directories ──────────────────────────────────
# When Drive is mounted these point to Drive — persistent across sessions.
# When Drive is unavailable they fall back to the local Colab SSD.
if USE_DRIVE:
    CKPT_DIR    = DRIVE_ROOT / "checkpoints"
    RESULTS_DIR = DRIVE_ROOT / "results"
    print(f"✓ Checkpoints / results → Google Drive")
else:
    CKPT_DIR    = DRIVE_ROOT / "checkpoints"
    RESULTS_DIR = DRIVE_ROOT / "results"
    print(f"✓ Checkpoints / results → local drive: {DRIVE_ROOT}")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("CKPT dir    :", CKPT_DIR)
print("Results dir :", RESULTS_DIR)

learn2learn imported from: /content/learn2learn/learn2learn/__init__.py
✓ Dataset cache → Google Drive: /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/data


Creating CIFARFS splits
cifarFS — train: 64 classes, val: 16 classes, test: 20 classes
Data stored at : /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/data
✓ Checkpoints / results → Google Drive
CKPT dir    : /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/checkpoints
Results dir : /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/results


## Section 3: Dataset & Episodic Loader

In [5]:
# ── Section 3: Dataset & Episodic Loader ─────────────────────────────────────
# Wraps the learn2learn datasets (loaded in Section 2) into a simple episodic
# sampler. Works with miniImageNet, tieredImageNet, CIFAR-FS, and CUB.
#
# Episodic protocol:
#   S = N_way × K_shot  support samples  (labeled)
#   Q = N_way × N_query query  samples  (unlabeled at inference)

import torch
import torch.nn.functional as F

# ── Generic episodic sampler wrapping any learn2learn dataset ─────────────────

class EpisodicSampler:
    """
    Wraps a learn2learn dataset (which exposes `labels_to_indices` dict)
    and samples N-way K-shot episodes on demand.

    Usage:
        sampler = EpisodicSampler(train_dataset, n_way=5, k_shot=1, n_query=15)
        s_imgs, s_lbl, q_imgs, q_lbl = sampler.sample()   # one episode

    Can also be called as a function (for compatibility with train/eval loops):
        s_imgs, s_lbl, q_imgs, q_lbl = sampler(n_way=5, k_shot=1, n_query=15)
    """

    def __init__(self, dataset, n_way: int = 5, k_shot: int = 1,
                 n_query: int = 15):
        self.dataset  = dataset
        self.n_way    = n_way
        self.k_shot   = k_shot
        self.n_query  = n_query

        # Build class → list[global_index] map
        if hasattr(dataset, 'labels_to_indices'):
            self.class_to_indices = {
                c: list(idxs)
                for c, idxs in dataset.labels_to_indices.items()
                if len(idxs) >= k_shot + n_query
            }
        elif hasattr(dataset, 'y'):
            import numpy as _np
            _labels = _np.array(dataset.y)
            self.class_to_indices = {
                int(c): list(_np.where(_labels == c)[0])
                for c in _np.unique(_labels)
                if len(_np.where(_labels == c)[0]) >= k_shot + n_query
            }
        else:
            raise ValueError("Dataset must expose .labels_to_indices or .y")

        self.classes = sorted(self.class_to_indices.keys())
        assert len(self.classes) >= n_way, (
            f"Only {len(self.classes)} classes available, N-way={n_way}")
        print(f"EpisodicSampler ready: {len(self.classes)} classes, "
              f"{n_way}-way {k_shot}-shot {n_query}-query")

    def sample(self):
        """Returns (support_imgs, support_labels, query_imgs, query_labels)."""
        episode_classes = random.sample(self.classes, self.n_way)
        s_imgs, s_labels, q_imgs, q_labels = [], [], [], []
        for local_lbl, cls in enumerate(episode_classes):
            pool = random.sample(self.class_to_indices[cls],
                                 self.k_shot + self.n_query)
            for i, idx in enumerate(pool):
                img, _ = self.dataset[idx]       # transform applied by dataset
                if not isinstance(img, torch.Tensor):
                    from torchvision.transforms.functional import to_tensor
                    img = to_tensor(img)
                if i < self.k_shot:
                    s_imgs.append(img);   s_labels.append(local_lbl)
                else:
                    q_imgs.append(img);   q_labels.append(local_lbl)
        return (torch.stack(s_imgs),  torch.tensor(s_labels, dtype=torch.long),
                torch.stack(q_imgs),  torch.tensor(q_labels, dtype=torch.long))

    def __call__(self, n_way=None, k_shot=None, n_query=None):
        """Allow calling as episode_fn(n_way, k_shot, n_query) for loop compat."""
        if n_way   is not None: self.n_way   = n_way
        if k_shot  is not None: self.k_shot  = k_shot
        if n_query is not None: self.n_query = n_query
        return self.sample()


# ── Instantiate samplers for all three splits ─────────────────────────────────
train_sampler = EpisodicSampler(train_dataset, n_way=5, k_shot=1, n_query=15)
val_sampler   = EpisodicSampler(val_dataset,   n_way=5, k_shot=1, n_query=15)
test_sampler  = EpisodicSampler(test_dataset,  n_way=5, k_shot=1, n_query=15)

# Augmented training sampler (same data, richer color/crop augmentation)
train_dataset_aug = DATASET_CLS[ACTIVE_DATASET](
    root=str(LOCAL_DATA_ROOT), mode='train',
    download=False, transform=AUG_TRANSFORM)
# FIX: Wrap train_dataset_aug with l2l.data.MetaDataset
train_dataset_aug = l2l.data.MetaDataset(train_dataset_aug)
train_sampler_aug = EpisodicSampler(train_dataset_aug, n_way=5, k_shot=1, n_query=15)

# ── Quick episode sanity check ────────────────────────────────────────────────
_s_imgs, _s_lbl, _q_imgs, _q_lbl = train_sampler.sample()
print(f"Support : {_s_imgs.shape}  labels={_s_lbl.tolist()}")
print(f"Query   : {_q_imgs.shape}  labels={_q_lbl[:5].tolist()}…")

# ── Synthetic planted-partition episode generator (Section 2.3) ───────────────

def synthetic_episode(n_way=5, n_nodes_per_class=5, feat_dim=64,
                      sigma=0.8, device=DEVICE):
    """
    Generate a planted-partition episode for topology-recovery experiments.
    Returns node features X, labels y, and ground-truth adjacency A_star.
    sigma controls noise: larger sigma → harder topology recovery.
    """
    centers = torch.randn(n_way, feat_dim)
    center_dists = [
        (centers[i] - centers[j]).norm().item()
        for i in range(n_way)
        for j in range(i + 1, n_way)
    ]
    avg_sep = sum(center_dists) / len(center_dists)

    N = n_way * n_nodes_per_class
    X = torch.zeros(N, feat_dim)
    y = torch.zeros(N, dtype=torch.long)
    for c in range(n_way):
        sl = slice(c * n_nodes_per_class, (c + 1) * n_nodes_per_class)
        X[sl] = centers[c] + sigma * avg_sep * torch.randn(n_nodes_per_class, feat_dim)
        y[sl] = c

    # Ground-truth: intra-class = 1, inter-class = 0, no self-loops
    A_star = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    A_star.fill_diagonal_(0)
    return X.to(device), y.to(device), A_star.to(device)


# ── Quick sanity check ────────────────────────────────────────────────────────
X_syn, y_syn, A_syn = synthetic_episode(n_way=5, n_nodes_per_class=4, feat_dim=64, sigma=0.6)
print(f"Synthetic X:{X_syn.shape}  y:{y_syn.shape}  A*:{A_syn.shape}")
print(f"Intra-class edges: {int(A_syn.sum().item())} "
      f"(expected {5 * 4 * 3} for 5 classes, 4 nodes each)")

EpisodicSampler ready: 64 classes, 5-way 1-shot 15-query
EpisodicSampler ready: 16 classes, 5-way 1-shot 15-query
EpisodicSampler ready: 20 classes, 5-way 1-shot 15-query
EpisodicSampler ready: 64 classes, 5-way 1-shot 15-query
Support : torch.Size([5, 3, 84, 84])  labels=[0, 1, 2, 3, 4]
Query   : torch.Size([75, 3, 84, 84])  labels=[0, 0, 0, 0, 0]…
Synthetic X:torch.Size([20, 64])  y:torch.Size([20])  A*:torch.Size([20, 20])
Intra-class edges: 60 (expected 60 for 5 classes, 4 nodes each)


## Section 4: Backbone Network (FSAKEBackbone)
Faithful copy of FSAKE's `EmbeddingImagenet` (FSAKE/model.py) — used unchanged for fair comparison per plan §2.2. Architecture: 4-conv with LeakyReLU(0.2), progressive channel widening 64→96→128→256, Dropout2d in layers 3/4, BN throughout, 128-dim output embedding.

In [ ]:
# ── Section 4: Backbone Network ───────────────────────────────────────────────
# FSAKEBackbone is a faithful copy of FSAKE's EmbeddingImagenet (model.py).
# Used as-is for fair comparison per plan §2.2:
#   "EVERYTHING must be identical to the baseline (backbone, feature dimension)
#    EXCEPT the graph construction and message passing."
#
# Architecture: 4-conv, LeakyReLU(0.2), progressive channel widening 64→96→128→256,
# Dropout2d in layers 3 & 4, BN after every conv, BN1d on final embedding.
# Input:  (B, 3, 84, 84)
# Output: (B, emb_size=128)
import torch.nn as nn


class FSAKEBackbone(nn.Module):
    """
    Backbone identical to FSAKE's EmbeddingImagenet (model.py).
    Preserved verbatim for fair comparison.
    """

    def __init__(self, emb_size: int = 128):
        super().__init__()
        hidden = 64
        self.hidden      = hidden
        self.last_hidden = hidden * 25   # spatial: 5×5 = 25 after 4 × MaxPool(2)
        self.emb_size    = emb_size

        self.conv_1 = nn.Sequential(
            nn.Conv2d(3, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.MaxPool2d(kernel_size=2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True))

        self.conv_2 = nn.Sequential(
            nn.Conv2d(hidden, int(hidden * 1.5), kernel_size=3, bias=False),  # no padding → 42→40
            nn.BatchNorm2d(int(hidden * 1.5)),
            nn.MaxPool2d(kernel_size=2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True))

        self.conv_3 = nn.Sequential(
            nn.Conv2d(int(hidden * 1.5), hidden * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden * 2),
            nn.MaxPool2d(kernel_size=2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Dropout2d(0.4))

        self.conv_4 = nn.Sequential(
            nn.Conv2d(hidden * 2, hidden * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden * 4),
            nn.MaxPool2d(kernel_size=2),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Dropout2d(0.5))

        # hidden*4 * 5*5 = 256 * 25 = 6400  →  emb_size
        self.layer_last = nn.Sequential(
            nn.Linear(self.last_hidden * 4, emb_size, bias=True),
            nn.BatchNorm1d(emb_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.conv_4(self.conv_3(self.conv_2(self.conv_1(x))))
        return self.layer_last(h.view(h.size(0), -1))


# Spatial size trace (sanity):
#   84×84 → MaxPool → 42×42          (conv_1, padding=1)
#   42×42 → k=3,no-pad → 40×40 → MaxPool → 20×20  (conv_2)
#   20×20 → MaxPool → 10×10          (conv_3, padding=1)
#   10×10 → MaxPool →  5×5           (conv_4, padding=1)
#   flatten: 256 × 5 × 5 = 6400  →  Linear  →  128

_bb = FSAKEBackbone(emb_size=128).to(DEVICE)
_x  = torch.randn(4, 3, 84, 84).to(DEVICE)
_bb.eval()
with torch.no_grad():
    _out = _bb(_x)
print(f"FSAKEBackbone output: {_out.shape}   (expected (4, 128))")
n_bb = sum(p.numel() for p in _bb.parameters())
print(f"Backbone params: {n_bb:,}")

Conv4 output shape: torch.Size([10, 128])
Conv4 trainable parameters: 317,760


## Section 5: Static k-NN Graph Module (HGNN Baseline — Ablation A1)

> **Important**: FSAKE does **not** use static k-NN. FSAKE already uses a dynamic per-layer scalar adjacency: `A^(l) = softmax(MLP(|h_i − h_j|))` (a 4-conv network on element-wise absolute differences). Static k-NN corresponds to the **HGNN** baseline (`A1_HGNN` in the ablation), which FSAKE already improved upon. Ablation `A2_FSAKE` uses `use_dynamic=True, use_edge_proj=False, use_case2=False` to approximate FSAKE's scalar-MLP-adjacency regime. DEKAE (`A5+`) replaces the scalar per-edge weight with a **vector** edge feature $e_{ij} \in \mathbb{R}^d$ and adds supervised edge-level loss.

In [ ]:
# ── Section 5: Static k-NN Graph Module (HGNN Baseline — Ablation A1) ────────
# Builds A ∈ R^{N×N} from pairwise Euclidean distances; retains top-k per node.
#
# NOTE: This is the HGNN (FSAKE predecessor) graph construction, NOT FSAKE.
# FSAKE recomputes a dynamic scalar adjacency at every GNN layer:
#   A^(l) = softmax(MLP(|h_i - h_j|))  ← 4-conv abs-diff network
# That dynamic scalar regime is approximated by ablation A2_FSAKE in Section 15.
# This static k-NN module is only used:
#   (a) as the A1_HGNN ablation baseline, and
#   (b) during the warm-up phase in DEKAE training (first t_warm epochs).

def build_knn_adjacency(node_feats: torch.Tensor, k: int,
                        symmetric: bool = True) -> torch.Tensor:
    """
    Args
    ----
    node_feats : (N, d) node embeddings
    k          : number of nearest neighbours per node
    symmetric  : if True, A = max(A, A^T) (undirected graph)

    Returns
    -------
    A : (N, N) float adjacency (0/1 values, no self-loops, row-normalised)
    """
    N = node_feats.size(0)
    # Pairwise squared Euclidean distance
    diff = node_feats.unsqueeze(0) - node_feats.unsqueeze(1)   # (N,N,d)
    dist = (diff ** 2).sum(-1)                                  # (N,N)

    # Mask self (set diagonal to large value so it is not selected in top-k)
    dist_masked = dist.clone()
    dist_masked.fill_diagonal_(float("inf"))

    # Keep k SMALLEST distances (nearest neighbours) → set all others to 0
    _, topk_idx = dist_masked.topk(k, dim=1, largest=False)
    A = torch.zeros(N, N, dtype=torch.float32, device=node_feats.device)
    A.scatter_(1, topk_idx, 1.0)

    if symmetric:
        A = torch.max(A, A.t())

    # Row-normalise (D^{-1} A)
    deg = A.sum(dim=1, keepdim=True).clamp(min=1e-6)
    A_norm = A / deg
    return A_norm


# ── Graph utility: density ────────────────────────────────────────────────────

def graph_density(A: torch.Tensor) -> float:
    """Fraction of possible directed edges that are non-zero (excl. diagonal)."""
    N = A.size(0)
    possible = N * (N - 1)
    actual   = (A > 0).float().sum().item() - (A.diagonal() > 0).float().sum().item()
    density  = actual / possible if possible > 0 else 0.0
    if density > 0.5:
        print(f"⚠  Graph density = {density:.3f} > 0.5 — risk of topology collapse!")
    return density


# ── Sanity check ─────────────────────────────────────────────────────────────
_feats = torch.randn(20, 128).to(DEVICE)
_A_knn = build_knn_adjacency(_feats, k=5)
d = graph_density(_A_knn)
print(f"k-NN adjacency shape: {_A_knn.shape}  density: {d:.3f}")

k-NN adjacency shape: torch.Size([20, 20])  density: 0.405


## Section 6: Edge Incidence Matrix & Dynamic Edge Features

**Key distinction from attention**: Attention produces a *scalar* weight per edge. Computing $E = BXW$ via the incidence matrix $B \in \{0,1\}^{N \times |E|}$ produces a *vector* $e_{ij} \in \mathbb{R}^d$ per edge — a $d$-dimensional signal retaining directional and heterogeneous relational information. This is equivalent to one step of line-graph convolution, where edges are first-class computational objects.

In [8]:
# ── Section 6: Edge Incidence Matrix & Dynamic Edge Features ─────────────────
# B ∈ {0,1}^{N × |E|}  maps each edge to its two incident nodes.
# E = B^T X W  →  each edge gets a vector representation in R^d.

class EdgeIncidenceModule(nn.Module):
    """
    Builds the edge incidence matrix B from a given adjacency A,
    then computes edge features E = B^T X W.

    Parameters
    ----------
    in_dim        : node feature dimension d
    edge_dim      : output edge feature dimension (default = in_dim)
    hidden        : hidden dim of the 2-layer edge MLP (≤ 64 to avoid overfit)
    edge_proj_type: 'mlp'    — 2-layer MLP (default, plan C3)
                    'linear' — single linear layer (plan C2)
                    'none'   — no learned projection; edge feature = zeros
                               (isolates adjacency-only contribution, plan C1)
    """

    def __init__(self, in_dim: int, edge_dim: int = None, hidden: int = 64,
                 edge_proj_type: str = "mlp"):
        super().__init__()
        edge_dim = edge_dim or in_dim
        self.edge_dim       = edge_dim
        self.edge_proj_type = edge_proj_type

        if edge_proj_type == "mlp":
            self.edge_mlp = nn.Sequential(
                nn.Linear(2 * in_dim, hidden, bias=True),
                nn.ReLU(inplace=True),
                nn.Linear(hidden, edge_dim, bias=True),
            )
        elif edge_proj_type == "linear":
            # Single linear projection — C2 ablation
            self.edge_mlp = nn.Linear(2 * in_dim, edge_dim, bias=True)
        elif edge_proj_type == "none":
            # No projection — edge features are zeros (C1 ablation)
            # DynamicTopologyModule falls back to bilinear node scores only;
            # EdgeToNodeProjection receives zero vectors → effectively disabled.
            self.edge_mlp = None
        else:
            raise ValueError(f"Unknown edge_proj_type: {edge_proj_type!r}. "
                             "Choose from 'mlp', 'linear', 'none'.")

    @staticmethod
    def adjacency_to_edge_index(A: torch.Tensor):
        """Return (src, dst) index arrays for all non-zero entries of A."""
        nz = A.nonzero(as_tuple=True)
        return nz[0], nz[1]         # src, dst

    def forward(self, X: torch.Tensor, A: torch.Tensor):
        """
        X : (N, d)  node features
        A : (N, N)  adjacency (may be weighted)

        Returns
        -------
        E         : (|E|, edge_dim)  vector edge features
                    (zero tensor when edge_proj_type='none')
        src, dst  : edge index tensors
        """
        src, dst = self.adjacency_to_edge_index(A)

        if self.edge_proj_type == "none":
            # C1 ablation: no learned edge features — return zeros.
            # DynamicTopologyModule will rely solely on bilinear node scores.
            # EdgeToNodeProjection receives zero vectors and contributes nothing;
            # the model degenerates to plain message passing over A'.
            E = torch.zeros(len(src), self.edge_dim,
                            device=X.device, dtype=X.dtype)
        else:
            # Concatenate incident node features for each edge
            edge_input = torch.cat([X[src], X[dst]], dim=-1)    # (|E|, 2d)
            E = self.edge_mlp(edge_input)                        # (|E|, edge_dim)
        return E, src, dst


# ── Sanity check ─────────────────────────────────────────────────────────────
_eim   = EdgeIncidenceModule(in_dim=128, edge_dim=128, hidden=64).to(DEVICE)
_feats = torch.randn(20, 128).to(DEVICE)
_A     = build_knn_adjacency(_feats, k=5)
_E, _src, _dst = _eim(_feats, _A)
print(f"Edge features shape : {_E.shape}   (|E|={len(_src)}, edge_dim=128)")
print(f"src index range     : {_src.min().item()} – {_src.max().item()}")

Edge features shape : torch.Size([136, 128])   (|E|=136, edge_dim=128)
src index range     : 0 – 19


## Section 7: Adaptive Topology Reconstruction

Low-rank bilinear scoring: $s_{ij} = (W_1 h_i)^\top (W_2 h_j)$ with rank $r \ll d$ bounds overfitting. Spectral normalization bounds $L_f$ (Lipschitz). Row-normalisation of $A'$ bounds $L_{MP}$. Together they ensure contraction (Proposition 3).

In [9]:
# ── Section 7: Adaptive Topology Reconstruction ───────────────────────────────
from torch.nn.utils.parametrizations import spectral_norm as spectral_norm_wrap

class DynamicTopologyModule(nn.Module):
    """
    Learns A'(l) = f(H(l)) using low-rank bilinear scoring.

    s_ij = (W1 h_i)^T (W2 h_j) / sqrt(r)     [low-rank, rank r << d]

    Spectral normalization on W1,W2 → Lipschitz bound on f.
    Row-normalization of A'          → Lipschitz bound on MP.

    sparsity_mode ∈ {'none', 'l1', 'topk', 'laplacian'}
    """

    def __init__(self, feat_dim: int, rank: int = 16, edge_dim: int = 128,
                 sparsity_mode: str = "l1", topk: int = 5,
                 lambda_sparse: float = 0.01, edge_dropout: float = 0.1):
        super().__init__()
        self.rank          = rank
        self.sparsity_mode = sparsity_mode
        self.topk          = topk
        self.lambda_sparse = lambda_sparse
        self.edge_dropout  = edge_dropout

        # Low-rank projection (spectral-normalised)
        self.W1 = spectral_norm_wrap(nn.Linear(feat_dim, rank, bias=False))
        self.W2 = spectral_norm_wrap(nn.Linear(feat_dim, rank, bias=False))

        # Edge → A' scoring via edge feature dot-product
        self.edge_scorer = spectral_norm_wrap(
            nn.Linear(edge_dim, 1, bias=False))

    def forward(self, H: torch.Tensor, E: torch.Tensor,
                src: torch.Tensor, dst: torch.Tensor,
                sparsity_reg: bool = True):
        """
        H   : (N, d)      node features
        E   : (|E|, d_e)  edge features from EdgeIncidenceModule
        src,dst : edge indices

        Returns
        -------
        A_prime       : (N, N)  new normalised adjacency
        sparsity_loss : scalar  (0 if sparsity_reg=False)
        """
        N = H.size(0)

        # ── Low-rank bilinear scores ─────────────────────────────────────────
        scores_bilinear = (self.W1(H[src]) * self.W2(H[dst])).sum(-1) / (self.rank ** 0.5)

        # ── Edge-feature scores ──────────────────────────────────────────────
        scores_edge     = self.edge_scorer(E).squeeze(-1)

        edge_scores = scores_bilinear + scores_edge

        # ── Soft adjacency via sigmoid ────────────────────────────────────────
        edge_weights = torch.sigmoid(edge_scores)

        # ── Dropout on edge weights during training ──────────────────────────
        # Applied AFTER sigmoid so that dropped edges get weight 0 (not 0.5).
        # Zeroing edge_scores before sigmoid would yield sigmoid(0)=0.5, meaning
        # "dropped" edges would still contribute — defeating the purpose.
        if self.training and self.edge_dropout > 0:
            drop_mask    = torch.bernoulli(
                torch.full_like(edge_weights, 1 - self.edge_dropout))
            edge_weights = edge_weights * drop_mask

        # ── Build dense A' ────────────────────────────────────────────────────
        A_prime = torch.zeros(N, N, device=H.device, dtype=H.dtype)
        A_prime[src, dst] = edge_weights

        # ── Sparsity regularisation ───────────────────────────────────────────
        sparsity_loss = torch.tensor(0.0, device=H.device)
        if sparsity_reg:
            if self.sparsity_mode == "l1":
                sparsity_loss = self.lambda_sparse * A_prime.abs().mean()
            elif self.sparsity_mode == "topk":
                # Hard top-k mask (gradient stops here — ablation only)
                vals, idx = A_prime.topk(self.topk, dim=1)
                mask = torch.zeros_like(A_prime)
                mask.scatter_(1, idx, 1.0)
                A_prime = A_prime * mask.detach()
            elif self.sparsity_mode == "laplacian":
                # tr(H^T L H) where L = D - A'
                D = A_prime.sum(dim=1).diag()
                L = D - A_prime
                sparsity_loss = self.lambda_sparse * torch.trace(H.t() @ L @ H)

        # ── Row-normalise (Proposition 3 Lipschitz control) ──────────────────
        deg = A_prime.sum(dim=1, keepdim=True).clamp(min=1e-6)
        A_prime = A_prime / deg

        return A_prime, sparsity_loss


def graph_density(A: torch.Tensor) -> float:
    """
    Quick graph density helper (used in early sanity checks before
    graph_metrics() is defined in Section 10).
    graph_density = |E| / (N*(N-1))  (self-loops excluded).
    """
    N = A.size(0)
    A_no_diag = A.clone()
    A_no_diag.fill_diagonal_(0)
    n_edges = (A_no_diag > 0).float().sum().item()
    return n_edges / max(N * (N - 1), 1)


# ── Sanity check ─────────────────────────────────────────────────────────────
_dtm   = DynamicTopologyModule(feat_dim=128, rank=16, edge_dim=128).to(DEVICE)
_A_dyn, _sl = _dtm(_feats, _E, _src, _dst)

print(f"Dynamic A' shape: {_A_dyn.shape}  sparsity_loss={_sl.item():.4f}")
print(f"Graph density   : {graph_density(_A_dyn):.3f}")   # uses helper defined above

Dynamic A' shape: torch.Size([20, 20])  sparsity_loss=0.0017
Graph density   : 0.329


## Section 8: Edge-Aware Node Update — Case 1 & Case 2

**Case 1 (node-aggregation with edge gating)** — `EdgeAwareKnowledgeFilter`:  
$X'_i = \sigma(g(e_i)) \cdot \sum_j A'_{ij} h_j$  
Edge features gate importance scores, but the node update is still a weighted sum of *neighboring node features*.

**Case 2 — the stronger novelty** — `EdgeToNodeProjection`:  
$$X' = A_{edge} \cdot E \cdot W', \quad A_{edge} \in \mathbb{R}^{N \times |E|}$$  
Each node is updated from **edge feature vectors** directly. $A_{edge}[i, e]$ holds a learned similarity weight for edge $e$ at its incident node $i$, computed as $\sigma(\text{MLP}([e_{ij} \| h_i]))$. This is structurally impossible with scalar attention and is equivalent to one step of line-graph message passing — exactly what novelty2.md calls the "stronger novelty" of Case 2.

Both modules share the same interface `(H, E, src, dst) → (scores, H_new)` so they are plug-in swappable via the `use_case2` flag in `DEKAEModel`.


In [10]:
# ── Section 8: Edge-Aware Node Update ────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# Case 1: EdgeAwareKnowledgeFilter
#   Node update = weighted sum of NEIGHBORING NODE FEATURES, gated by edge agg.
#   X'_i = σ(g(e_i)) · Σ_j A'_ij h_j
# ─────────────────────────────────────────────────────────────────────────────
class EdgeAwareKnowledgeFilter(nn.Module):
    """
    Case 1: classic node-aggregation gated by aggregated edge features.
    Kept for ablation baseline (A3 path).
    """

    def __init__(self, edge_dim: int, node_dim: int, hidden: int = 64):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(edge_dim + node_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 1),
        )

    def forward(self, H: torch.Tensor, E: torch.Tensor,
                src: torch.Tensor, dst: torch.Tensor,
                A_prime: torch.Tensor):
        """
        H       : (N, d)
        E       : (|E|, edge_dim)
        src,dst : edge endpoint indices
        A_prime : (N, N) row-normalised dynamic adjacency

        Returns
        -------
        scores   : (N,) importance scores in [0,1]
        H_pooled : (N, d) node update via A' @ H, gated by edge aggregation
        """
        N        = H.size(0)
        edge_dim = E.size(1)

        # Mean-aggregate incident edge features per node
        agg = torch.zeros(N, edge_dim, device=H.device)
        agg.scatter_add_(0, src.unsqueeze(-1).expand_as(E), E)
        counts = torch.zeros(N, device=H.device).scatter_add_(
            0, src, torch.ones(len(src), device=H.device))
        agg = agg / counts.unsqueeze(-1).clamp(min=1)

        gate_input = torch.cat([agg, H], dim=-1)
        scores     = torch.sigmoid(self.gate(gate_input)).squeeze(-1)  # (N,)

        H_pooled   = A_prime @ H                             # Case 1: node→node
        H_pooled   = H_pooled * scores.unsqueeze(-1)
        return scores, H_pooled


# ─────────────────────────────────────────────────────────────────────────────
# Case 2: EdgeToNodeProjection  ← THE STRONGER NOVELTY (novelty2.md §4, Case 2)
#   Node update = weighted sum of EDGE FEATURE VECTORS
#   X' = A_edge @ E @ W'   where A_edge ∈ R^{N × |E|}
#
#   A_edge[i, e] = σ( MLP([e_ij ‖ h_i]) )  — learned similarity-based weight
#                                              non-zero only when i ∈ {src_e, dst_e}
#
#   Structural difference from Case 1 / attention:
#     • Message source is a VECTOR e_{ij} ∈ R^d, not a scalar weight
#     • Equivalent to one step of line-graph message passing
#     • Enables edge-level loss supervision (structurally impossible with scalars)
# ─────────────────────────────────────────────────────────────────────────────
class EdgeToNodeProjection(nn.Module):
    """
    Case 2: X' = A_edge @ E @ W'

    For each edge e = (i, j):
      - Compute assignment weight w_i = σ( scorer([e ‖ h_i]) )  at node i (src)
      - Compute assignment weight w_j = σ( scorer([e ‖ h_j]) )  at node j (dst)
      - Scatter: H'_i += w_i · (E_e @ W');  H'_j += w_j · (E_e @ W')

    The result H'_i is a weighted sum of *projected edge feature vectors*,
    not of neighboring node features — this is the key structural distinction.

    Parameters
    ----------
    edge_dim : edge feature dimension (from EdgeIncidenceModule)
    node_dim : output node feature dimension
    """

    def __init__(self, edge_dim: int, node_dim: int):
        super().__init__()
        # W': edge_dim → node_dim  (the projection matrix in X' = A_edge E W')
        self.W_proj = nn.Linear(edge_dim, node_dim, bias=False)

        # Similarity scorer: how much does edge e "belong" to its incident node i
        # Input: [e_ij ‖ h_i]  →  scalar assignment weight
        self.sim_scorer = nn.Linear(edge_dim + node_dim, 1, bias=True)

        # Output gate (per-node, acts as importance score for FSAKE compatibility)
        self.out_gate = nn.Sequential(
            nn.Linear(node_dim, node_dim, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, H: torch.Tensor, E: torch.Tensor,
                src: torch.Tensor, dst: torch.Tensor,
                A_prime: torch.Tensor = None):
        """
        H       : (N, d)         node features
        E       : (|E|, edge_dim) edge feature vectors
        src,dst : (|E|,)         edge endpoint indices
        A_prime : unused (accepted for interface compatibility)

        Returns
        -------
        scores : (N,)  per-node importance proxy (mean of output gate activations)
        H_new  : (N, d) node features updated from edge vectors  X' = A_edge E W'
        """
        N      = H.size(0)
        E_proj = self.W_proj(E)                                    # (|E|, d)

        # ── Learned assignment weights at each endpoint ───────────────────────
        # w_src[e] = σ( scorer( [E[e] ‖ H[src[e]]] ) )
        w_src = torch.sigmoid(
            self.sim_scorer(torch.cat([E, H[src]], dim=-1))
        ).squeeze(-1)                                              # (|E|,)
        w_dst = torch.sigmoid(
            self.sim_scorer(torch.cat([E, H[dst]], dim=-1))
        ).squeeze(-1)                                              # (|E|,)

        # ── Scatter: H'_i = Σ_{e incident to i} w_ei · (E_e W') ─────────────
        H_new = torch.zeros(N, E_proj.size(1), device=H.device, dtype=H.dtype)
        H_new.scatter_add_(0,
                           src.unsqueeze(-1).expand_as(E_proj),
                           E_proj * w_src.unsqueeze(-1))
        H_new.scatter_add_(0,
                           dst.unsqueeze(-1).expand_as(E_proj),
                           E_proj * w_dst.unsqueeze(-1))

        # ── Normalize by total incident weight per node ───────────────────────
        weight_sum = torch.zeros(N, device=H.device, dtype=H.dtype)
        weight_sum.scatter_add_(0, src, w_src)
        weight_sum.scatter_add_(0, dst, w_dst)
        H_new = H_new / weight_sum.unsqueeze(-1).clamp(min=1e-6)  # (N, d)

        # ── Output gate ───────────────────────────────────────────────────────
        gate   = self.out_gate(H_new)                              # (N, d)
        scores = gate.mean(dim=-1)                                 # (N,) proxy
        H_new  = H_new * gate                                      # (N, d)

        return scores, H_new


# ── Sanity checks ─────────────────────────────────────────────────────────────
# Case 1
_kf = EdgeAwareKnowledgeFilter(edge_dim=128, node_dim=128).to(DEVICE)
_scores_c1, _H_c1 = _kf(_feats, _E, _src, _dst, _A_dyn)
print("── Case 1 (EdgeAwareKnowledgeFilter) ──")
print(f"  scores shape : {_scores_c1.shape}  range [{_scores_c1.min():.3f}, {_scores_c1.max():.3f}]")
print(f"  H_new  shape : {_H_c1.shape}")

# Case 2
_e2n = EdgeToNodeProjection(edge_dim=128, node_dim=128).to(DEVICE)
_scores_c2, _H_c2 = _e2n(_feats, _E, _src, _dst)
print("\n── Case 2 (EdgeToNodeProjection)  X' = A_edge @ E @ W' ──")
print(f"  scores shape : {_scores_c2.shape}  range [{_scores_c2.min():.3f}, {_scores_c2.max():.3f}]")
print(f"  H_new  shape : {_H_c2.shape}")

# Verify output differs from Case 1 (structural difference, not just parameterisation)
diff = (_H_c1 - _H_c2).abs().mean().item()
print(f"\n  Mean |Case1 − Case2| = {diff:.4f}  (expected > 0 — structurally different)")


── Case 1 (EdgeAwareKnowledgeFilter) ──
  scores shape : torch.Size([20])  range [0.442, 0.549]
  H_new  shape : torch.Size([20, 128])

── Case 2 (EdgeToNodeProjection)  X' = A_edge @ E @ W' ──
  scores shape : torch.Size([20])  range [0.498, 0.500]
  H_new  shape : torch.Size([20, 128])

  Mean |Case1 − Case2| = 0.1706  (expected > 0 — structurally different)


## Section 9: Supervised Edge Loss (Support-Support Only)

**Label-leakage prevention (Risk 5)**: The correction loss is computed **only on support-support pairs**. Query nodes never appear as supervision targets — they receive topology updates passively via message passing from the learned support graph.

$$\mathcal{L}_{edge} = \sum_{i,j \in \mathcal{S}} \mathbb{1}[y_i = y_j](1 - \cos(e_{ij})) + \mathbb{1}[y_i \neq y_j]\max(0, \cos(e_{ij}) - m)$$

In [11]:
# ── Section 9: Supervised Edge Loss (Support-Support Only) ───────────────────

def edge_correction_loss(E: torch.Tensor,
                         src: torch.Tensor,
                         dst: torch.Tensor,
                         H: torch.Tensor,
                         labels: torch.Tensor,
                         support_mask: torch.Tensor,
                         margin: float = 0.5) -> torch.Tensor:
    """
    Computes the edge-level contrastive correction loss.

    Similarity is computed as F.cosine_similarity(H[src_ss], H[dst_ss]) --
    comparing raw node embeddings at each edge's endpoints rather than
    splitting concatenated edge features (which was incorrect).

    IMPORTANT: only support→support edges are used.
    An assertion guarantees no query indices leak into this computation.

    Parameters
    ----------
    E            : (|E|, d)  edge features from EdgeIncidenceModule
                   (kept for API consistency; not used in similarity)
    src, dst     : edge endpoint indices (for ALL edges in the graph)
    H            : (N, d) node feature matrix (post-backbone, pre-GNN)
    labels       : (N,) — labels for support nodes; query labels NOT used
    support_mask : (N,) bool — True if node i is in the support set
    margin       : contrastive margin m

    Returns
    -------
    loss : scalar
    """
    # ── Strict label-leakage guard ────────────────────────────────────────────
    # Keep only support-support edges
    ss_mask = support_mask[src] & support_mask[dst]

    if ss_mask.sum().item() == 0:
        return torch.tensor(0.0, device=E.device, requires_grad=True)

    src_ss = src[ss_mask]
    dst_ss = dst[ss_mask]

    # ASSERT: every endpoint of a selected edge is truly a support node.
    # The previous check (ss_mask.sum <= N_s^2) was trivially true and never
    # caught actual leakage.  This check is the meaningful guard.
    assert support_mask[src_ss].all() and support_mask[dst_ss].all(), \
        "Edge loss applied to non-support edges — label leakage!"

    # Node cosine similarity: compare the GNN-input embeddings of each edge's
    # endpoints. This measures how similar the two connected nodes are in the
    # representation space learned by the backbone (E is available for future use).
    cos_sim = F.cosine_similarity(H[src_ss], H[dst_ss], dim=-1)

    same_class = (labels[src_ss] == labels[dst_ss]).float()

    pull_loss = same_class       * (1 - cos_sim)
    push_loss = (1 - same_class) * torch.clamp(cos_sim - margin, min=0)
    return (pull_loss + push_loss).mean()

# ── Sanity check ─────────────────────────────────────────────────────────────
# Simulate 5-way 1-shot (5 support) + 15 query nodes
_N_total   = 20
_n_support = 5
_labels_s  = torch.arange(_n_support).to(DEVICE)              # support labels
_labels    = torch.cat([_labels_s,
                        torch.zeros(_N_total - _n_support, dtype=torch.long).to(DEVICE)])
_sup_mask  = torch.zeros(_N_total, dtype=torch.bool).to(DEVICE)
_sup_mask[:_n_support] = True

_loss_edge = edge_correction_loss(_E, _src, _dst, _feats, _labels, _sup_mask)
print(f"Edge correction loss (support-only): {_loss_edge.item():.4f}")

Edge correction loss (support-only): 0.0000


## Section 10: Sparsity Regularization

Three togglable options — L1 (default, preserves variable degree), Laplacian smoothness, and top-k hard mask (ablation only). `graph_density()` raises a warning above 0.5 to detect collapse.

In [12]:
# ── Section 10: Sparsity Regularization ──────────────────────────────────────
# Standalone helpers (also used inside DynamicTopologyModule).

def l1_sparsity_loss(A: torch.Tensor, lambda_: float = 0.01) -> torch.Tensor:
    """L1 regularisation on adjacency. Promotes sparse connectivity."""
    return lambda_ * A.abs().mean()


def laplacian_smoothness_loss(A: torch.Tensor, H: torch.Tensor,
                               lambda_: float = 0.01) -> torch.Tensor:
    """Laplacian smoothness: tr(H^T L H). Encourages connected nodes to agree."""
    D = A.sum(dim=1).diag()
    L = D - A
    return lambda_ * torch.trace(H.t() @ L @ H)


def topk_mask(A: torch.Tensor, k: int) -> torch.Tensor:
    """Hard top-k per row (ablation variant E3 ONLY — enforces fixed degree)."""
    # NOTE: gradient does not flow through this mask.
    _, idx = A.topk(k, dim=1)
    mask = torch.zeros_like(A)
    mask.scatter_(1, idx, 1.0)
    return A * mask.detach()


def graph_metrics(A: torch.Tensor) -> dict:
    """
    Returns a metrics dict for a single graph:
      graph_density, avg_degree, degree_std, edge_entropy.
    Raises a warning if density > 0.5.
    """
    N   = A.size(0)
    # Remove diagonal for degree counting
    A_no_diag = A.clone()
    A_no_diag.fill_diagonal_(0)

    degree    = (A_no_diag > 0).float().sum(dim=1)      # (N,)
    n_edges   = (A_no_diag > 0).float().sum().item()
    density   = n_edges / max(N * (N - 1), 1)

    # Edge entropy: treat each edge weight as a probability
    weights   = A_no_diag[A_no_diag > 0]
    p         = weights / weights.sum().clamp(min=1e-9)
    edge_ent  = -(p * (p + 1e-9).log()).sum().item()

    if density > 0.5:
        print(f"⚠  graph_density={density:.3f} > 0.5 — potential collapse")

    return {
        "graph_density": density,
        "avg_degree"   : degree.mean().item(),
        "degree_std"   : degree.std().item(),
        "edge_entropy" : edge_ent,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
_metrics = graph_metrics(_A_dyn)
print("Graph metrics:", {k: round(v, 4) for k, v in _metrics.items()})
print("L1 loss      :", l1_sparsity_loss(_A_dyn).item())

Graph metrics: {'graph_density': 0.3289, 'avg_degree': 6.25, 'degree_std': 2.3141, 'edge_entropy': 4.7454}
L1 loss      : 0.0004999999655410647


## Section 11: Full Model Assembly (DEKAE)

Forward pass per GNN layer: `node features + B → edge features E → dynamic A' → node update → skip connection`.

**Case 1 path** (`use_case2=False`): `H' = σ(g(E)) · (A' @ H)` — node features aggregated, gated by edge summaries.  
**Case 2 path** (`use_case2=True`, default): `H' = A_edge @ E @ W'` — nodes updated directly from edge feature *vectors*, the true novelty from novelty2.md.

Warm-up schedule: static k-NN for first `T_warm` epochs. Curriculum `λ_edge`: ramps 0→`lambda_edge` over first 20% of training.


In [ ]:
# ── Section 11: Full Model Assembly (DEKAE) ───────────────────────────────────
import math

# ── Latent Mediator Transformer (Dependencies) ───────────────────────────────

class LatentMediatorLayer(nn.Module):
    """
    Single Latent Mediator Transformer layer.
    Phase A (Gather): M ← CrossAttn(Q=M, KV=[Q‖S])
    Phase B (Distribute): Q ← CrossAttn(Q=Q, KV=M), S ← CrossAttn(Q=S, KV=M)
    """
    def __init__(self, embed_dim: int, n_heads: int = 4, dropout: float = 0.1,
                 phases: str = "both"):
        super().__init__()
        self.phases = phases

        self.gather_attn   = nn.MultiheadAttention(embed_dim, n_heads,
                                                    dropout=dropout,
                                                    batch_first=True)
        self.gather_norm   = nn.LayerNorm(embed_dim)
        self.gather_ff     = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.gather_ff_norm = nn.LayerNorm(embed_dim)

        self.distrib_q_attn = nn.MultiheadAttention(embed_dim, n_heads,
                                                     dropout=dropout,
                                                     batch_first=True)
        self.distrib_q_norm = nn.LayerNorm(embed_dim)

        self.distrib_s_attn = nn.MultiheadAttention(embed_dim, n_heads,
                                                     dropout=dropout,
                                                     batch_first=True)
        self.distrib_s_norm = nn.LayerNorm(embed_dim)

    def forward(self, M, Q, S):
        if self.phases in ("both", "gather_only"):
            context   = torch.cat([Q, S], dim=1)
            M_a, _    = self.gather_attn(M, context, context)
            M         = self.gather_norm(M + M_a)
            M         = self.gather_ff_norm(M + self.gather_ff(M))

        if self.phases in ("both", "distribute_only"):
            Q_a, _    = self.distrib_q_attn(Q, M, M)
            Q         = self.distrib_q_norm(Q + Q_a)
            S_a, _    = self.distrib_s_attn(S, M, M)
            S         = self.distrib_s_norm(S + S_a)

        return M, Q, S


class LatentMediatorTransformer(nn.Module):
    """Iterative Latent Mediator Transformer for cross-set knowledge exchange."""
    def __init__(self, embed_dim: int, n_mediators: int = 8,
                 n_layers: int = 3, n_heads: int = 4, dropout: float = 0.1,
                 init_strategy: str = "learned", phases: str = "both"):
        super().__init__()
        self.embed_dim      = embed_dim
        self.n_mediators    = n_mediators
        self.init_strategy  = init_strategy
        self.phases         = phases

        _m = torch.empty(1, n_mediators, embed_dim)
        nn.init.trunc_normal_(_m, std=0.02)
        if init_strategy == "learned":
            self.mediator_init = nn.Parameter(_m)
        elif init_strategy == "fixed":
            self.register_buffer("mediator_init", _m)
        elif init_strategy == "zero":
            self.register_buffer("mediator_init",
                                 torch.zeros(1, n_mediators, embed_dim))
        else:
            raise ValueError(f"Unknown init_strategy: {init_strategy!r}")

        self.layers = nn.ModuleList([
            LatentMediatorLayer(embed_dim, n_heads, dropout, phases=phases)
            for _ in range(n_layers)
        ])
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.s_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, H_query, H_support):
        Q = H_query.unsqueeze(0)
        S = H_support.unsqueeze(0)
        M = self.mediator_init.expand(Q.size(0), -1, -1)

        for layer in self.layers:
            M, Q, S = layer(M, Q, S)

        Q_out = self.q_proj(Q.squeeze(0))
        S_out = self.s_proj(S.squeeze(0))
        return Q_out, S_out


class DEKAEModel(nn.Module):
    """Dynamic Edge-driven topology for Episodic few-shot learning (DEKAE)."""

    def __init__(self, embed_dim: int = 128, n_gnn_layers: int = 3,
                 rank: int = 16, sparsity_mode: str = "l1",
                 lambda_sparse: float = 0.01, lambda_edge: float = 0.5,
                 topk_k: int = 5, n_way: int = 5, use_dynamic: bool = True,
                 use_edge_proj: bool = True, use_case2: bool = True,
                 edge_dropout: float = 0.1,
                 use_lmt: bool = True, n_mediators: int = 8,
                 lmt_layers: int = 3, lmt_heads: int = 4, lmt_dropout: float = 0.1,
                 lmt_init_strategy: str = "learned", lmt_phases: str = "both",
                 edge_loss_mode: str = "all",
                 adj_residual: float = 0.15,
                 edge_proj_type: str = "mlp",
                 selection_strategy: str = "mean"):
        super().__init__()
        self.embed_dim        = embed_dim
        self.n_gnn_layers     = n_gnn_layers
        self.n_way            = n_way
        self.use_dynamic      = use_dynamic
        self.use_edge_proj    = use_edge_proj
        self.use_case2        = use_case2
        self.use_lmt          = use_lmt
        self.edge_loss_mode   = edge_loss_mode
        self.adj_residual     = adj_residual
        self.knn_k            = topk_k
        self.lambda_edge      = lambda_edge
        self.edge_proj_type   = edge_proj_type
        self.selection_strategy = selection_strategy

        # FSAKEBackbone is identical to FSAKE's EmbeddingImagenet — fair comparison
        self.backbone = FSAKEBackbone(emb_size=embed_dim)

        self.edge_modules = nn.ModuleList([
            EdgeIncidenceModule(embed_dim, embed_dim, hidden=64,
                                edge_proj_type=edge_proj_type)
            for _ in range(n_gnn_layers)
        ])
        self.dynamic_modules = nn.ModuleList([
            DynamicTopologyModule(embed_dim, rank, embed_dim,
                                  sparsity_mode, topk_k, lambda_sparse,
                                  edge_dropout)
            for _ in range(n_gnn_layers)
        ])
        # Case 2: EdgeToNodeProjection (default)
        self.e2n_modules = nn.ModuleList([
            EdgeToNodeProjection(embed_dim, embed_dim)
            for _ in range(n_gnn_layers)
        ])
        # Case 1: EdgeAwareKnowledgeFilter (ablation A3/A4)
        self.kf_modules = nn.ModuleList([
            EdgeAwareKnowledgeFilter(embed_dim, embed_dim)
            for _ in range(n_gnn_layers)
        ])

        if use_lmt:
            self.lmt = LatentMediatorTransformer(
                embed_dim      = embed_dim,
                n_mediators    = n_mediators,
                n_layers       = lmt_layers,
                n_heads        = lmt_heads,
                dropout        = lmt_dropout,
                init_strategy  = lmt_init_strategy,
                phases         = lmt_phases,
            )
        else:
            self.lmt = None

        self.classifier = nn.Linear(embed_dim, n_way)

    def set_use_dynamic(self, flag: bool):
        self.use_dynamic = flag

    def set_use_edge_proj(self, flag: bool):
        """False → ablation A1/A2 (plain A'@H)."""
        self.use_edge_proj = flag

    def set_use_case2(self, flag: bool):
        self.use_case2 = flag

    def set_use_lmt(self, flag: bool):
        self.use_lmt = flag

    def set_selection_strategy(self, strategy: str):
        """Group B: 'mean' | 'nearest_centroid' | 'max'."""
        assert strategy in ("mean", "nearest_centroid", "max"), \
            f"Unknown selection_strategy {strategy!r}."
        self.selection_strategy = strategy

    def forward(self, imgs_support: torch.Tensor, labels_support: torch.Tensor,
                imgs_query: torch.Tensor, lambda_edge_scale: float = 1.0):
        N_s = imgs_support.size(0)
        N_q = imgs_query.size(0)

        all_imgs = torch.cat([imgs_support, imgs_query], dim=0)
        H0       = self.backbone(all_imgs)

        support_mask = torch.zeros(N_s + N_q, dtype=torch.bool, device=H0.device)
        support_mask[:N_s] = True

        H = H0
        total_sparsity_loss = torch.tensor(0.0, device=H0.device)
        total_edge_loss     = torch.tensor(0.0, device=H0.device)
        layer_stability     = []
        intra_edge_ratio    = 0.0
        inter_edge_ratio    = 0.0

        # Fallback adjacency for n_gnn_layers=0 (A0_ProtoNet: pure backbone)
        A_prime = build_knn_adjacency(H0, k=self.knn_k)

        for l in range(self.n_gnn_layers):
            A_init = build_knn_adjacency(H, k=self.knn_k)
            E, src, dst = self.edge_modules[l](H, A_init)

            if self.use_dynamic:
                A_prime, sp_loss = self.dynamic_modules[l](H, E, src, dst)

                if self.adj_residual > 0:
                    A_prime = (1.0 - self.adj_residual) * A_prime \
                              + self.adj_residual * A_init

                total_sparsity_loss = total_sparsity_loss + sp_loss / self.n_gnn_layers

                labels_full = torch.full((N_s + N_q,), -1,
                                         dtype=torch.long, device=H.device)
                labels_full[:N_s] = labels_support
                e_loss_l = edge_correction_loss(E, src, dst, H, labels_full,
                                                support_mask)

                if self.edge_loss_mode == "penultimate":
                    if l == self.n_gnn_layers - 1:
                        total_edge_loss = total_edge_loss + e_loss_l * lambda_edge_scale
                elif self.edge_loss_mode in ("ascending", "decaying"):
                    asc_w = 1.0 / max(self.n_gnn_layers - l, 1)
                    total_edge_loss = total_edge_loss + e_loss_l * lambda_edge_scale * asc_w
                else:
                    total_edge_loss = total_edge_loss + e_loss_l * lambda_edge_scale

                if l == self.n_gnn_layers - 1:
                    with torch.no_grad():
                        ss_mask = support_mask[src] & support_mask[dst]
                        if ss_mask.sum() > 0:
                            ss_src = src[ss_mask]; ss_dst = dst[ss_mask]
                            same = (labels_full[ss_src] == labels_full[ss_dst]).float()
                            intra_edge_ratio = same.mean().item()
                            inter_edge_ratio = 1.0 - intra_edge_ratio
            else:
                A_prime = A_init   # static (warm-up / A1_HGNN / A3_static_edge)

            if self.use_edge_proj:
                if self.use_case2:
                    # Case 2: X' = A_edge @ E @ W'  (DEKAE novelty)
                    _, H_new = self.e2n_modules[l](H, E, src, dst)
                else:
                    # Case 1 (A3/A4): X' = σ(g(E)) · A' @ H
                    _, H_new = self.kf_modules[l](H, E, src, dst, A_prime)
            else:
                # A1/A2 (use_edge_proj=False): plain A'@H
                H_new = A_prime @ H

            H_new = H_new + H   # skip connection

            with torch.no_grad():
                stability = (H_new - H).norm().item()
            layer_stability.append(stability)
            H = H_new

        H_support = H[:N_s]
        H_query   = H[N_s:]

        if self.use_lmt and self.lmt is not None:
            H_query, H_support = self.lmt(H_query, H_support)

        # ── Prototype computation (Group B ablation: mean / max / nearest_centroid)
        prototypes = torch.zeros(self.n_way, self.embed_dim, device=H.device)
        for c in range(self.n_way):
            mask_c = labels_support == c
            if mask_c.sum() > 0:
                cands = H_support[mask_c]
                if self.selection_strategy == "max":
                    # Max-pool prototype: max across support samples per dim (Group B1/B3)
                    prototypes[c] = cands.max(dim=0).values
                elif (self.selection_strategy == "nearest_centroid"
                      and cands.size(0) > 1):
                    rough = cands.mean(dim=0, keepdim=True)
                    sim = F.cosine_similarity(
                        cands, rough.expand(cands.size(0), -1), dim=-1)
                    weights = F.softmax(sim * 5.0, dim=0)
                    prototypes[c] = (cands * weights.unsqueeze(1)).sum(dim=0)
                else:
                    prototypes[c] = cands.mean(dim=0)

        H_q_norm   = F.normalize(H_query.clamp(-1e4, 1e4),    dim=-1, eps=1e-8)
        proto_norm = F.normalize(prototypes.clamp(-1e4, 1e4), dim=-1, eps=1e-8)
        logits     = H_q_norm @ proto_norm.t()

        aux_loss = total_edge_loss * self.lambda_edge + total_sparsity_loss

        metrics = graph_metrics(A_prime)
        metrics["layer_stability"]  = layer_stability
        metrics["intra_edge_ratio"] = intra_edge_ratio
        metrics["inter_edge_ratio"] = inter_edge_ratio

        return logits, aux_loss, metrics


# ── Sanity checks ─────────────────────────────────────────────────────────────

_model = DEKAEModel(n_way=5, embed_dim=128, use_case2=True,
                    use_lmt=True, n_mediators=8, lmt_layers=3).to(DEVICE)
_s_imgs = torch.randn(5, 3, 84, 84).to(DEVICE)
_s_lbl  = torch.arange(5).to(DEVICE)
_q_imgs = torch.randn(15, 3, 84, 84).to(DEVICE)
_logits, _aloss, _mets = _model(_s_imgs, _s_lbl, _q_imgs)
print("── Full model (Case 2 + LMT) ──")
print(f"  Logits : {_logits.shape}  Aux loss : {_aloss.item():.4f}")
print(f"  Density: {_mets['graph_density']:.3f}  Intra-edge: {_mets['intra_edge_ratio']:.3f}")
n_param = sum(p.numel() for p in _model.parameters())
print(f"  Total params: {n_param:,}")

_model_proto = DEKAEModel(n_way=5, embed_dim=128, n_gnn_layers=0,
                           use_dynamic=False, use_edge_proj=False,
                           use_case2=False, use_lmt=False).to(DEVICE)
_logits_proto, _aloss_proto, _ = _model_proto(_s_imgs, _s_lbl, _q_imgs)
print(f"\n── A0_ProtoNet (n_gnn_layers=0) — logits: {_logits_proto.shape}  aux={_aloss_proto.item():.4f}")

_model_a1 = DEKAEModel(n_way=5, embed_dim=128,
                        use_dynamic=False, use_edge_proj=False,
                        use_case2=False, use_lmt=False).to(DEVICE)
_logits_a1, _, _ = _model_a1(_s_imgs, _s_lbl, _q_imgs)
print(f"── A1_HGNN (static k-NN, plain A@H) — logits: {_logits_a1.shape}")

_model_a2 = DEKAEModel(n_way=5, embed_dim=128,
                        use_edge_proj=False, use_lmt=False).to(DEVICE)
_logits_a2, _, _ = _model_a2(_s_imgs, _s_lbl, _q_imgs)
print(f"── A2_FSAKE (dynamic, no edge proj) — logits: {_logits_a2.shape}")

_model_max = DEKAEModel(n_way=5, embed_dim=128, use_lmt=False,
                         selection_strategy="max").to(DEVICE)
_logits_max, _, _ = _model_max(_s_imgs, _s_lbl, _q_imgs)
print(f"── B1_max (max-pool prototype) — logits: {_logits_max.shape}")

── Full model (Case 2 + LMT) ──
  Logits : torch.Size([15, 5])  Aux loss : 0.4550
  Density: 0.384  Intra-edge: 0.000
  Total params: 1,580,683

── Ablation A3 (Case 1, no LMT) — logits: torch.Size([15, 5])
── Ablation A2 (dynamic only, no edge proj) — logits: torch.Size([15, 5])


## Section 11.5: Latent Mediator Transformer (LMT)

Instead of directly computing cosine similarity between raw query and support prototypes, the **Latent Mediator Transformer** uses a small set of learnable *mediator tokens* ($\mathbf{M} \in \mathbb{R}^{m \times d}$, $m \ll N_q, N_s$) to broker cross-set communication through two phases in each layer:

**Phase A — Gather** (mediator aggregates from both sets):
$$\mathbf{M}^{(l+1)} = \text{CrossAttn}\!\left(\mathbf{Q}=\mathbf{M}^{(l)},\ \mathbf{K}\!=\!\mathbf{V}=[\mathbf{Q}^{(l)} \| \mathbf{S}^{(l)}]\right)$$

**Phase B — Distribute** (each set updates from the mediator):
$$\mathbf{Q}^{(l+1)} = \text{CrossAttn}\!\left(\mathbf{Q}=\mathbf{Q}^{(l)},\ \mathbf{K}\!=\!\mathbf{V}=\mathbf{M}^{(l+1)}\right)$$
$$\mathbf{S}^{(l+1)} = \text{CrossAttn}\!\left(\mathbf{Q}=\mathbf{S}^{(l)},\ \mathbf{K}\!=\!\mathbf{V}=\mathbf{M}^{(l+1)}\right)$$

**Complexity**: $O(m(N_q + N_s)d)$ per layer — far cheaper than direct $O(N_q N_s d)$ cross-attention, and the bottleneck acts as a natural knowledge compressor.

The mediator tokens are **learnable** (shared meta-knowledge across episodes) and evolve iteratively across $L_{lmt}$ layers, producing increasingly discriminative relational summaries.


In [14]:
# ── Section 11.5: Latent Mediator Transformer — Sanity Checks ────────────────
# LatentMediatorLayer and LatentMediatorTransformer are defined in Section 11
# (the cell above DEKAEModel).  This cell provides complexity analysis and
# extended ablation sanity checks.

# ── Complexity sanity check ───────────────────────────────────────────────────
def lmt_complexity(N_q, N_s, m, d):
    """Compare LMT vs direct cross-attention FLOPs (approximate)."""
    lmt_flops    = m * (N_q + N_s) * d
    direct_flops = N_q * N_s * d
    return {"lmt": lmt_flops, "direct_cross_attn": direct_flops,
            "reduction_factor": direct_flops / max(lmt_flops, 1)}

print("Complexity comparison (5-way 1-shot + 15 query, d=128, m=8):")
print(lmt_complexity(N_q=15, N_s=5, m=8, d=128))

# ── Module sanity checks ──────────────────────────────────────────────────────
_lmt = LatentMediatorTransformer(embed_dim=128, n_mediators=8,
                                  n_layers=3, n_heads=4,
                                  init_strategy="learned", phases="both").to(DEVICE)
_H_q = torch.randn(15, 128).to(DEVICE)
_H_s = torch.randn(5,  128).to(DEVICE)
_H_q_r, _H_s_r = _lmt(_H_q, _H_s)
print(f"LMT (default, learned+both) query: {_H_q_r.shape}  support: {_H_s_r.shape}")
print(f"LMT params: {sum(p.numel() for p in _lmt.parameters()):,}")

# H4: fixed init (no gradient through mediator tokens)
_lmt_fixed = LatentMediatorTransformer(embed_dim=128, n_mediators=8,
                                        n_layers=3, n_heads=4,
                                        init_strategy="fixed").to(DEVICE)
_H_q_f, _ = _lmt_fixed(_H_q, _H_s)
print(f"LMT (fixed init) query: {_H_q_f.shape}")

# H5: gather-only phase
_lmt_gather = LatentMediatorTransformer(embed_dim=128, n_mediators=8,
                                         n_layers=3, n_heads=4,
                                         phases="gather_only").to(DEVICE)
_H_q_g, _ = _lmt_gather(_H_q, _H_s)
# gather_only: Phase B is skipped so Q/S are not updated from the mediator;
# q_proj/s_proj are still applied for consistent architecture — output ≠ raw input.
print(f"LMT (gather_only) query: {_H_q_g.shape}  (no mediator→Q broadcast; q_proj still applied)")


Complexity comparison (5-way 1-shot + 15 query, d=128, m=8):
{'lmt': 20480, 'direct_cross_attn': 9600, 'reduction_factor': 0.46875}
LMT (default, learned+both) query: torch.Size([15, 128])  support: torch.Size([5, 128])
LMT params: 1,026,688
LMT (fixed init) query: torch.Size([15, 128])
LMT (gather_only) query: torch.Size([15, 128])  (no mediator→Q broadcast; q_proj still applied)


## Section 12: Training Loop with Checkpointing

In [15]:
# ── Section 12a: Training Health Monitor ─────────────────────────────────────
# Fires after every eval epoch inside train().
# Prints a ⚠ line for each triggered condition so problems are visible
# immediately in the Colab output without having to inspect raw logs.

def _health_check(log: dict, history: list, epoch: int, cfg: dict):
    """
    Inspect the latest epoch log and recent history and print actionable
    warnings when training shows known failure patterns.

    Conditions checked
    ------------------
    1. Stuck at chance          — val_acc ≤ chance+0.01 after warm-up
    2. Train-val divergence     — train_acc >> val_acc (overfitting early)
    3. Loss not decreasing      — train_loss barely moved over last 10 epochs
    4. Vanishing gradients      — grad_norm_mean < 1e-4
    5. Exploding gradients      — grad_norm_mean > 5.0  (pre-clip norm reported by
                                   clip_grad_norm_; a large value means gradients were
                                   explosive before clipping)
    6. Graph density collapse   — already caught in train(), repeated here
    7. Intra-edge not rising    — edge loss active but intra-edge ratio < 0.4
       after ramp window.
       NOTE: SKIPPED when k_shot == 1. In 5-way 1-shot there is exactly one support
       node per class, so no two support nodes can share the same label. The
       support-support intra_edge_ratio is therefore structurally 0.000 by
       definition — it is NOT a sign that edge supervision has failed.
       This check is only meaningful for k_shot ≥ 2, where same-class support
       pairs exist and the edge loss can pull them together.
    8. All NaN                  — nan_skipped equals n_episodes_train
       (every episode was poisoned)
    9. Topology frozen          — topology_stability ≈ 0 while dynamic is on
       (dynamic module learned nothing — weights all near zero).
       Only flagged when episodes were NOT all NaN-skipped, to avoid
       false positives caused by an empty ep_densities list.
    """
    chance     = 1.0 / cfg.get("n_way", 5)   # derived from config, not hardcoded
    past_warm  = epoch > cfg.get("t_warm", 5)
    ramp_done  = epoch > cfg.get("t_warm", 5) + cfg.get("edge_loss_ramp", 20)
    n_episodes = cfg.get("n_episodes_train", 300)
    k_shot     = cfg.get("k_shot", 1)
    flags      = []

    val_acc   = log.get("val_acc", 0)
    train_acc = log.get("train_acc", 0)
    gnorm     = log.get("grad_norm_mean", 1.0)
    density   = log.get("graph_density", 1.0)
    intra     = log.get("intra_edge_ratio", 0)
    topo_std  = log.get("topology_stability", 1.0)
    nan_skip  = log.get("nan_skipped", 0)
    train_loss = log.get("train_loss", 0)

    # 1. Stuck at chance after warm-up
    if past_warm and val_acc <= chance + 0.01:
        flags.append(
            f"val_acc={val_acc:.3f} ≤ chance ({chance:.3f}) — model not learning. "
            "Check: data pipeline, label alignment, LR (try 1e-3), "
            "or set use_dynamic=False to test backbone alone."
        )

    # 2. Train-val divergence (overfitting)
    if past_warm and (train_acc - val_acc) > 0.20:
        flags.append(
            f"train_acc={train_acc:.3f} >> val_acc={val_acc:.3f} (+{train_acc-val_acc:.2f}) "
            "— overfitting. Try: lower LR, higher weight_decay, reduce n_gnn_layers."
        )

    # 3. Loss plateau (compare to 10 epochs ago if available)
    if len(history) >= 10:
        old_loss = history[-10]["train_loss"]
        if abs(old_loss - train_loss) < 0.002:
            flags.append(
                f"train_loss plateaued at {train_loss:.4f} for 10 epochs. "
                "Try: increase LR, check that optimizer.step() is reached "
                "(nan_skipped not too high), verify data is varied."
            )

    # 4. Vanishing gradients
    if gnorm < 1e-4:
        flags.append(
            f"grad_norm_mean={gnorm:.2e} — vanishing. "
            "Check: spectral norm on DynTopo, activation functions, "
            "or skip connections are wired correctly."
        )

    # 5. Exploding gradients (pre-clip norm — clip_grad_norm_ returns the original norm)
    if gnorm > 5.0:
        flags.append(
            f"grad_norm_mean={gnorm:.2f} (pre-clip) — explosive gradients. "
            "Try: lower grad_clip (0.5), lower LR, check for unbounded activations."
        )

    # 6. Density collapse (redundant with train() warning, shown here too)
    if log.get("use_dynamic", True) and density < 0.01:
        flags.append(
            f"graph_density={density:.4f} — topology collapsed. "
            "Lower lambda_sparse further or increase adj_residual above 0.15."
        )

    # 7. Intra-edge ratio not rising after edge-loss ramp
    # Skip in 1-shot: with 1 support node per class there are zero same-class
    # support-support pairs by definition, so intra_edge_ratio is always 0.000.
    # This is mathematically expected, NOT a training failure.
    if ramp_done and k_shot >= 2 and intra < 0.40:
        flags.append(
            f"intra_edge_ratio={intra:.3f} after edge-loss ramp — edge supervision "
            "not separating classes. Check edge_correction_loss or lower lambda_edge."
        )

    # 8. All episodes were NaN-skipped
    if nan_skip >= n_episodes * 0.9:
        flags.append(
            f"nan_skipped={nan_skip}/{n_episodes} — nearly all updates poisoned. "
            "Training is not progressing. Restart with lower lambda_sparse and LR."
        )

    # 9. Topology frozen (dynamic on but zero variance across episodes).
    # Guard: skip when nearly all episodes were NaN-skipped — ep_densities would be
    # empty in that case, making topology_stability=0.0 a misleading false positive.
    if (log.get("use_dynamic", True) and past_warm
            and topo_std < 1e-5
            and nan_skip < n_episodes * 0.9):
        flags.append(
            f"topology_stability={topo_std:.2e} — dynamic module outputs constant graph. "
            "DynamicTopologyModule weights may be near zero. Check initialisation."
        )

    if flags:
        print(f"  ── Health check epoch {epoch} ──────────────────────────────")
        for f in flags:
            print(f"  ⚠ {f}")
        print(f"  ───────────────────────────────────────────────────────────")
    # No output when all checks pass (silent green)


print("_health_check() registered — fires every eval_every epochs during training.")

_health_check() registered — fires every eval_every epochs during training.


In [ ]:
import time, json, re as _re
from torch.optim import Adam
from tqdm.notebook import tqdm

# ── Hyperparameters ────────────────────────────────────────────────────────────
# Fair-comparison note (plan §2.2):
#   lr, weight_decay, grad_clip match FSAKE's train.py exactly.
#   Scheduler mirrors FSAKE's halve-every-10k-iterations policy.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "n_way"          : 5,
    "k_shot"         : 1,
    "n_query"        : 15,
    "embed_dim"      : 128,
    "n_gnn_layers"   : 3,
    "rank"           : 16,
    "sparsity_mode"  : "l1",    # 'l1' | 'topk' | 'none'  (laplacian excluded — harmful)
    "lambda_sparse"  : 0.001,
    "lambda_edge"    : 0.05,
    "topk_k"         : 5,
    "lr"             : 1e-3,    # FSAKE default (train.py: tt.arg.lr = 1e-3)
    "weight_decay"   : 1e-6,    # FSAKE default (train.py: tt.arg.weight_decay = 1e-6)
    "n_episodes_train": 300,    # episodes per epoch
    "n_epochs"       : 200,
    "t_warm"         : 5,       # warm-up epochs with static k-NN
    "edge_loss_ramp" : 20,      # epochs over which lambda_edge ramps to final
    "grad_clip"      : 5.0,     # FSAKE default (train.py: tt.arg.grad_clip = 5)
    "eval_every"     : 5,
    "n_eval_episodes": 200,
    "seed"           : 42,
    "use_edge_proj"  : True,
    "use_case2"      : True,
    "use_lmt"        : True,
    "use_dynamic"    : True,
    "n_mediators"    : 8,
    "lmt_layers"     : 3,
    "lmt_heads"      : 4,
    "lmt_init_strategy": "learned",
    "lmt_phases"     : "both",
    "edge_loss_mode" : "all",
    "ckpt_every"     : 10,
}


def build_model(cfg: dict) -> DEKAEModel:
    """Constructs a DEKAEModel from a config dict."""
    return DEKAEModel(
        embed_dim     = cfg["embed_dim"],
        n_gnn_layers  = cfg["n_gnn_layers"],
        rank          = cfg["rank"],
        sparsity_mode = cfg["sparsity_mode"],
        lambda_sparse = cfg["lambda_sparse"],
        lambda_edge   = cfg["lambda_edge"],
        topk_k        = cfg["topk_k"],
        n_way         = cfg["n_way"],
        use_dynamic        = cfg.get("use_dynamic", True),
        use_edge_proj      = cfg.get("use_edge_proj", True),
        use_case2          = cfg.get("use_case2", True),
        use_lmt            = cfg.get("use_lmt", True),
        n_mediators        = cfg.get("n_mediators", 8),
        lmt_layers         = cfg.get("lmt_layers", 3),
        lmt_heads          = cfg.get("lmt_heads", 4),
        lmt_init_strategy  = cfg.get("lmt_init_strategy", "learned"),
        lmt_phases         = cfg.get("lmt_phases", "both"),
        edge_loss_mode     = cfg.get("edge_loss_mode", "all"),
        edge_proj_type     = cfg.get("edge_proj_type", "mlp"),
        selection_strategy = cfg.get("selection_strategy", "mean"),
    ).to(DEVICE)


def run_episode(model: DEKAEModel, episode_fn, n_way, k_shot, n_query,
                lambda_edge_scale: float = 1.0):
    """Runs one episode; returns (loss, accuracy, metrics_dict)."""
    s_imgs, s_lbl, q_imgs, q_lbl = episode_fn(n_way, k_shot, n_query)
    s_imgs = s_imgs.to(DEVICE)
    s_lbl  = s_lbl.to(DEVICE)
    q_imgs = q_imgs.to(DEVICE)
    q_lbl  = q_lbl.to(DEVICE)

    logits, aux_loss, metrics = model(s_imgs, s_lbl, q_imgs, lambda_edge_scale)

    cls_loss = F.cross_entropy(logits, q_lbl)
    total    = cls_loss + aux_loss

    preds = logits.argmax(dim=1)
    acc   = (preds == q_lbl).float().mean().item()
    return total, acc, metrics


def train(
    cfg: dict,
    episode_fn_train,
    episode_fn_val,
    run_name: str = "dekae",
    log_wandb: bool = False,
    start_epoch: int = 0,
    initial_history: list = None,
    resume_model_state: dict = None,
    resume_optimizer_state: dict = None,
    resume_scheduler_state: dict = None,
) -> tuple:
    """
    Full episodic training loop for DEKAE.

    Parameters
    ----------
    cfg                    : hyperparameter config dict (CFG)
    episode_fn_train       : callable(n_way, k_shot, n_query) → episode tensors
    episode_fn_val         : callable(n_way, k_shot, n_query) → episode tensors
    run_name               : prefix for checkpoint / history filenames
    log_wandb              : whether to push metrics to Weights & Biases
    start_epoch            : epoch to resume from (0 = fresh start)
    initial_history        : pre-loaded epoch log list from a prior run
    resume_model_state     : state_dict loaded from an epoch checkpoint
    resume_optimizer_state : optimizer state_dict from an epoch checkpoint
    resume_scheduler_state : scheduler state_dict from an epoch checkpoint

    Returns (trained_model, history).
    """
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    model    = build_model(cfg)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {n_params:,}")

    optimizer = Adam(model.parameters(),
                     lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    # ── LR schedule: halve every dec_lr_step epochs ───────────────────────────
    # Mirrors FSAKE's policy: lr *= 0.5^(iter // 10000) over 100k iterations.
    # Converting to epoch units: 10000 iters / n_episodes_train ≈ 33 epochs.
    _dec_lr_step = max(1, round(10000 / cfg.get("n_episodes_train", 300)))
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=_dec_lr_step, gamma=0.5)

    if resume_model_state is not None:
        model.load_state_dict(resume_model_state)
        print(f"  ↩  Resumed model weights from epoch {start_epoch}")
    if resume_optimizer_state is not None:
        optimizer.load_state_dict(resume_optimizer_state)
        print(f"  ↩  Resumed optimizer state")
    if resume_scheduler_state is not None:
        scheduler.load_state_dict(resume_scheduler_state)
        print(f"  ↩  Resumed LR scheduler state")

    history      = list(initial_history) if initial_history else []
    best_val_acc = max((e["val_acc"] for e in history if e["val_acc"] > 0), default=0.0)
    if start_epoch > 0:
        print(f"  Resuming from epoch {start_epoch + 1} / {cfg['n_epochs']}  "
              f"(best val so far: {best_val_acc:.4f})")

    ckpt_path      = CKPT_DIR    / f"{run_name}_best.pth"
    history_path   = RESULTS_DIR / f"{run_name}_history.json"
    ckpt_every     = cfg.get("ckpt_every", 10)

    ramp_epochs = cfg.get("edge_loss_ramp", 20)
    t_warm      = cfg.get("t_warm", 5)
    eval_every  = cfg.get("eval_every", 5)
    grad_clip   = cfg.get("grad_clip", 5.0)

    for epoch in range(start_epoch + 1, cfg["n_epochs"] + 1):
        t0 = time.time()

        # Respect cfg["use_dynamic"]: A1_HGNN/A0_ProtoNet must stay static.
        _want_dynamic = cfg.get("use_dynamic", True)
        model.set_use_dynamic(_want_dynamic and (epoch > t_warm))

        lambda_edge_scale = min(1.0, (epoch - t_warm) / max(ramp_epochs, 1))
        lambda_edge_scale = max(0.0, lambda_edge_scale)

        model.train()
        train_accs, train_losses = [], []
        ep_densities, ep_intra   = [], []
        nan_skipped              = 0
        gnorms                   = []

        for _ in range(cfg["n_episodes_train"]):
            loss, acc, mets = run_episode(
                model, episode_fn_train,
                cfg["n_way"], cfg["k_shot"], cfg["n_query"],
                lambda_edge_scale,
            )

            if torch.isnan(loss):
                nan_skipped += 1
                continue

            optimizer.zero_grad()
            loss.backward()
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            train_accs.append(acc)
            train_losses.append(loss.item())
            gnorms.append(gnorm.item() if hasattr(gnorm, "item") else float(gnorm))
            ep_densities.append(mets.get("graph_density", 0.0))
            ep_intra.append(mets.get("intra_edge_ratio", 0.0))

        scheduler.step()

        train_acc  = float(np.mean(train_accs))   if train_accs   else 0.0
        train_loss = float(np.mean(train_losses))  if train_losses else float("nan")
        density    = float(np.mean(ep_densities))  if ep_densities else 0.0
        intra      = float(np.mean(ep_intra))      if ep_intra     else 0.0
        gnorm_mean = float(np.mean(gnorms))        if gnorms       else 0.0

        log = {
            "epoch"             : epoch,
            "train_acc"         : train_acc,
            "train_loss"        : train_loss,
            "val_acc"           : 0.0,
            "val_ci"            : 0.0,
            "graph_density"     : density,
            "intra_edge_ratio"  : intra,
            "nan_skipped"       : nan_skipped,
            "grad_norm_mean"    : gnorm_mean,
            "topology_stability": float(np.std(ep_densities)) if ep_densities else 0.0,
        }

        if epoch % eval_every == 0 or epoch == cfg["n_epochs"]:
            model.eval()
            val_accs = []
            with torch.no_grad():
                for _ in range(cfg["n_eval_episodes"]):
                    _, v_acc, _ = run_episode(
                        model, episode_fn_val,
                        cfg["n_way"], cfg["k_shot"], cfg["n_query"],
                        lambda_edge_scale=1.0,
                    )
                    val_accs.append(v_acc)

            val_acc = float(np.mean(val_accs))
            val_ci  = 1.96 * float(np.std(val_accs)) / (len(val_accs) ** 0.5)
            log["val_acc"] = val_acc
            log["val_ci"]  = val_ci

            elapsed = time.time() - t0

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save({
                    "epoch"     : epoch,
                    "state"     : model.state_dict(),
                    "cfg"       : cfg,
                    "val_acc"   : val_acc,
                    "optimizer" : optimizer.state_dict(),
                }, str(ckpt_path))
                try:
                    import os as _os_ck
                    with open(str(ckpt_path), 'rb') as _fsync_fd:
                        _os_ck.fsync(_fsync_fd.fileno())
                except Exception:
                    pass
                print(f"  ✓ New best val acc = {val_acc:.4f}  (best ckpt saved)")

            intra_str = "N/A(1-shot)" if cfg.get("k_shot", 1) < 2 else f"{intra:.3f}"
            print(
                f"Epoch {epoch:>4} | train_acc={train_acc:.4f} "
                f"val_acc={val_acc:.4f}±{val_ci:.4f} "
                f"density={density:.3f} intra={intra_str} "
                f"nan_skip={nan_skipped} t={elapsed:.1f}s"
            )

            _health_check(log, history, epoch, cfg)

            if log_wandb:
                try:
                    import wandb
                    wandb.log(log)
                except Exception:
                    pass

        history.append(log)

        if epoch % ckpt_every == 0:
            _ep_ckpt = CKPT_DIR / f"{run_name}_epoch_{epoch}.pth"
            torch.save({
                "epoch"    : epoch,
                "state"    : model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "cfg"      : cfg,
                "history"  : history,
            }, str(_ep_ckpt))
            try:
                import os as _os_ck
                with open(str(_ep_ckpt), 'rb') as _fsync_fd:
                    _os_ck.fsync(_fsync_fd.fileno())
            except Exception:
                pass
            print(f"  💾 Epoch checkpoint → {_ep_ckpt.name}")
            try:
                with open(str(history_path), "w") as f:
                    json.dump(history, f, indent=2)
            except Exception:
                pass

    try:
        with open(str(history_path), "w") as f:
            json.dump(history, f, indent=2)
    except Exception:
        pass

    return model, history

print(f"build_model / run_episode / train defined — CFG ready.")
print(f"  lr={CFG['lr']}  weight_decay={CFG['weight_decay']}  grad_clip={CFG['grad_clip']}")
print(f"  scheduler: StepLR (step_size≈{max(1,round(10000/CFG['n_episodes_train']))} epochs, gamma=0.5)")

build_model / run_episode / train defined — CFG ready.
  Periodic epoch checkpoints every 10 epochs → /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/checkpoints


## Section 12b: Execute Training (CIFAR-FS, 5-way 1-shot)

In [17]:
# ── Colab Keep-Alive (optional) ──────────────────────────────────────────────
# Prevents Colab from disconnecting due to idle timeout during long runs.
# This uses a JavaScript trick that simulates a button click every 60 s.
# It does NOT prevent hard compute-unit exhaustion — only idle-timeout disconnects.
# Run this cell BEFORE the training cell, then run training normally.

try:
    from IPython.display import display, Javascript
    display(Javascript("""
function keepAlive() {
    var intervalId = setInterval(function() {
        try {
            document.querySelector('#toolbar-container #run-all-above').click();
        } catch(e) {}
        try {
            document.querySelector('.run-button').click();
        } catch(e) {}
        console.log('[DEKAE keep-alive] ping at ' + new Date().toLocaleTimeString());
    }, 60000);  // every 60 seconds
    console.log('[DEKAE keep-alive] started — interval id: ' + intervalId);
}
keepAlive();
"""))
    print("✓ Keep-alive script injected (runs every 60 s in the browser tab).")
    print("  Keep your browser tab active and visible for best results.")
except Exception as _e:
    print(f"⚠ Keep-alive not available ({_e}).")


<IPython.core.display.Javascript object>

✓ Keep-alive script injected (runs every 60 s in the browser tab).
  Keep your browser tab active and visible for best results.


In [ ]:
# ── Section 12b: Execute Training ────────────────────────────────────────────
# Supports automatic resume from Google Drive epoch checkpoints.
# On the FIRST run: starts training from scratch.
# After a runtime reset: re-run ALL cells top-to-bottom; the cell below
#   automatically finds the latest epoch_{N}.pth on Drive and resumes from N+1.
#
# Estimated time on Colab T4 GPU:
#   200 epochs × 300 episodes ≈ 120–150 min total
#
# ── Optional: enable W&B cloud logging ───────────────────────────────────────
# Uncomment the two lines below to sync metrics to Weights & Biases.
# init_wandb("dekae_miniImageNet_5way1shot")
# LOG_WANDB = True

LOG_WANDB = False   # set True after calling init_wandb()

# ── Training config (edit here to override CFG) ──────────────────────────────
# CFG is already defined in Section 12. Override individual keys here if needed.
# To do a quick iteration: CFG["n_epochs"] = 50

# ── Auto-resume: detect latest epoch checkpoint on Drive ─────────────────────
import re as _re_train

def _find_latest_epoch_ckpt(ckpt_dir, run_name: str):
    """
    Scan ckpt_dir for files matching  <run_name>_epoch_<N>.pth
    and return (epoch_number, path) for the highest N found, or (0, None).
    """
    candidates = list(ckpt_dir.glob(f"{run_name}_epoch_*.pth"))
    if not candidates:
        return 0, None
    def _epoch_num(p):
        m = _re_train.search(r"epoch_(\d+)", p.name)
        return int(m.group(1)) if m else -1
    best = max(candidates, key=_epoch_num)
    return _epoch_num(best), best

_resumed_epoch, _epoch_ckpt_path = _find_latest_epoch_ckpt(CKPT_DIR, "dekae_miniImageNet")
_resume_kwargs = {}

if _epoch_ckpt_path is not None and _resumed_epoch < CFG["n_epochs"]:
    print(f"↩  Found epoch checkpoint at epoch {_resumed_epoch}: {_epoch_ckpt_path.name}")
    print(f"   Resuming training from epoch {_resumed_epoch + 1} / {CFG['n_epochs']}")
    _ckpt = torch.load(str(_epoch_ckpt_path), map_location=DEVICE)
    _resume_kwargs = dict(
        start_epoch            = _ckpt["epoch"],
        initial_history        = _ckpt.get("history", []),
        resume_model_state     = _ckpt["state"],
        resume_optimizer_state = _ckpt["optimizer"],
        resume_scheduler_state = _ckpt.get("scheduler"),
    )
elif _epoch_ckpt_path is not None and _resumed_epoch >= CFG["n_epochs"]:
    print(f"✓  Training already completed ({_resumed_epoch} / {CFG['n_epochs']} epochs).")
    print(f"   Loading best checkpoint for evaluation cells …")
    _best_ckpt_path = CKPT_DIR / "dekae_miniImageNet_best.pth"
    if _best_ckpt_path.exists():
        _best_ckpt = torch.load(str(_best_ckpt_path), map_location=DEVICE)
        trained_model = build_model(CFG)
        trained_model.load_state_dict(_best_ckpt["state"])
        trained_model.eval()
        # Restore history from last epoch checkpoint
        _last_ckpt = torch.load(str(_epoch_ckpt_path), map_location=DEVICE)
        train_history = _last_ckpt.get("history", [])
        best_val = max((e["val_acc"] for e in train_history if e["val_acc"] > 0), default=0.0)
        print(f"   Best val accuracy: {best_val * 100:.2f}%")
    # Skip the train() call below since training is already done
    _resume_kwargs = None
else:
    print("No epoch checkpoint found — starting training from scratch.")

# ── Print training configuration ─────────────────────────────────────────────
if _resume_kwargs is not None:
    print("=" * 60)
    print("DEKAE Training — miniImageNet  5-way 1-shot")
    print("=" * 60)
    print(f"  epochs          : {CFG['n_epochs']}")
    print(f"  episodes/epoch  : {CFG['n_episodes_train']}")
    print(f"  warm-up epochs  : {CFG['t_warm']}")
    print(f"  edge-loss ramp  : {CFG['edge_loss_ramp']} epochs")
    print(f"  sparsity mode   : {CFG['sparsity_mode']}  (lambda_sparse={CFG['lambda_sparse']})")
    print(f"  lambda_edge     : {CFG['lambda_edge']}")
    print(f"  ckpt_every      : {CFG['ckpt_every']} epochs  (→ {CKPT_DIR})")
    print(f"  scheduler       : StepLR (step={max(1, round(10000 / CFG['n_episodes_train']))} epochs, gamma=0.5)  [matches FSAKE]")
    print(f"  device          : {DEVICE}")
    print("=" * 60)

    trained_model, train_history = train(
        cfg              = CFG,
        episode_fn_train = train_sampler_aug,    # augmented miniImageNet train split
        episode_fn_val   = val_sampler,          # miniImageNet val split
        run_name         = "dekae_miniImageNet",
        log_wandb        = LOG_WANDB,
        **_resume_kwargs,
    )

    best_val = max((e["val_acc"] for e in train_history if e["val_acc"] > 0), default=0.0)
    print(f"\n✓ Training complete — best val accuracy: {best_val * 100:.2f}%")
    print(f"  Best checkpoint → {CKPT_DIR}/dekae_miniImageNet_best.pth")
    print(f"  History         → {RESULTS_DIR}/dekae_miniImageNet_history.json")


No epoch checkpoint found — starting training from scratch.
DEKAE Training — CIFAR-FS  5-way 1-shot
  epochs          : 200
  episodes/epoch  : 300
  warm-up epochs  : 5
  edge-loss ramp  : 20 epochs
  sparsity mode   : l1  (lambda_sparse=0.001)
  lambda_edge     : 0.05
  ckpt_every      : 10 epochs  (→ /content/drive/.shortcut-targets-by-id/1dTRmYHasNbSCArFCb5fKSToqp0gwiSCb/DEKAE_Project/checkpoints)
  scheduler       : CosineAnnealingLR (T_max=200, eta_min=1e-6)
  device          : cuda
Model parameters: 1,580,683
  ✓ New best val acc = 0.2971  (best ckpt saved)
Epoch    5 | train_acc=0.3103 val_acc=0.2971±0.0119 density=0.084 intra=N/A(1-shot) nan_skip=0 t=321.7s
  ✓ New best val acc = 0.3205  (best ckpt saved)
Epoch   10 | train_acc=0.3185 val_acc=0.3205±0.0134 density=0.086 intra=N/A(1-shot) nan_skip=0 t=314.9s
  💾 Epoch checkpoint → dekae_cifarfs_epoch_10.pth
  ✓ New best val acc = 0.3335  (best ckpt saved)
Epoch   15 | train_acc=0.3367 val_acc=0.3335±0.0127 density=0.091 intra=N

## Section 13: Evaluation & Statistical Testing

In [ ]:
# ── Section 13: Evaluation & Statistical Testing ─────────────────────────────
from scipy import stats as scipy_stats

def evaluate(model: DEKAEModel, episode_fn, n_way: int, k_shot: int,
             n_query: int, n_episodes: int = 1000) -> dict:
    """
    Evaluate over n_episodes test episodes.
    Returns accuracy mean, 95% CI, and full per-episode result dict.
    """
    model.eval()
    accs, densities, avg_degrees = [], [], []
    intra_ratios, inter_ratios   = [], []
    # intra_ratios tracked explicitly for Group D avg_intra_ratio reporting

    with torch.no_grad():
        for _ in tqdm(range(n_episodes), desc="Evaluating"):
            s_imgs, s_lbl, q_imgs, q_lbl = episode_fn(n_way, k_shot, n_query)
            s_imgs = s_imgs.to(DEVICE)
            s_lbl  = s_lbl.to(DEVICE)
            q_imgs = q_imgs.to(DEVICE)
            q_lbl  = q_lbl.to(DEVICE)

            logits, _, mets = model(s_imgs, s_lbl, q_imgs)
            preds = logits.argmax(dim=1)
            acc   = (preds == q_lbl).float().mean().item()
            accs.append(acc)
            densities.append(mets["graph_density"])
            avg_degrees.append(mets["avg_degree"])
            intra_ratios.append(mets.get("intra_edge_ratio", 0.0))

    accs_np  = np.array(accs)
    mean_acc = accs_np.mean()
    ci_95    = 1.96 * accs_np.std() / (len(accs_np) ** 0.5)

    return {
        "mean_acc"             : mean_acc,
        "ci_95"                : ci_95,
        "per_episode"          : accs,
        "avg_density"          : np.mean(densities),
        "avg_degree"           : np.mean(avg_degrees),
        "per_episode_density"  : densities,   # Group D: topology_stability per seed
        "avg_intra_ratio"      : np.mean(intra_ratios) if intra_ratios else 0.0,
    }


def paired_ttest(accs_a: list, accs_b: list) -> dict:
    """
    Paired t-test between two models across the same episodes.
    Returns t-statistic and p-value. p < 0.05 → significant improvement.
    """
    t_stat, p_val = scipy_stats.ttest_rel(accs_b, accs_a)
    return {"t_statistic": t_stat, "p_value": p_val,
            "significant_at_0.05": p_val < 0.05}


def compute_intra_inter_edge_ratio(A: torch.Tensor, labels: torch.Tensor):
    """
    Given an adjacency matrix and node labels, compute the ratio of
    intra-class and inter-class edges (for graph quality reporting).
    """
    N = A.size(0)
    intra = inter = 0
    for i in range(N):
        for j in range(N):
            if i != j and A[i, j] > 0:
                if labels[i] == labels[j]:
                    intra += 1
                else:
                    inter += 1
    total = intra + inter
    return intra / max(total, 1), inter / max(total, 1)


# ── Example usage (illustrative — replace dummy episode_fn with real loader) ──
def _dummy_episode_fn(n_way, k_shot, n_query):
    """Placeholder using synthetic data — replace with real dataset loader."""
    s_imgs  = torch.randn(n_way * k_shot,  3, 84, 84)
    s_lbl   = torch.repeat_interleave(torch.arange(n_way), k_shot)
    q_imgs  = torch.randn(n_way * n_query, 3, 84, 84)
    q_lbl   = torch.repeat_interleave(torch.arange(n_way), n_query)
    return s_imgs, s_lbl, q_imgs, q_lbl

_eval_result = evaluate(_model, _dummy_episode_fn, n_way=5, k_shot=1,
                        n_query=15, n_episodes=50)
print(f"Accuracy: {_eval_result['mean_acc']*100:.2f} ± "
      f"{_eval_result['ci_95']*100:.2f}% (95% CI)")
print(f"Avg density: {_eval_result['avg_density']:.3f}")

In [ ]:
# ── Section 13b: Real Test-Set Evaluation ────────────────────────────────────
# Run AFTER Section 12b training completes.
# Evaluates the best checkpoint on the miniImageNet test split.
# Runs 600 episodes for stable 95% CI (plan requirement: >= 600 episodes).

import os

# ── Load best checkpoint if available ────────────────────────────────────────
ckpt_path = CKPT_DIR / "dekae_miniImageNet_best.pth"
if os.path.exists(str(ckpt_path)):
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    eval_model = build_model(ckpt["cfg"])
    eval_model.load_state_dict(ckpt["state"])
    eval_model.set_use_dynamic(True)
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}  "
          f"(val_acc={ckpt['val_acc']*100:.2f}%)")
else:
    # No checkpoint yet — use the in-memory trained model from Section 12b
    eval_model = trained_model
    print("No checkpoint found — using in-memory trained_model.")

# ── 5-way 1-shot on test split (600 episodes) ────────────────────────────────
print("\nRunning 5-way 1-shot evaluation on miniImageNet test split (600 episodes)…")
test_result_1shot = evaluate(eval_model, test_sampler,
                             n_way=5, k_shot=1, n_query=15,
                             n_episodes=600)
print(f"\n  5-way  1-shot: {test_result_1shot['mean_acc']*100:.2f} "
      f"+/- {test_result_1shot['ci_95']*100:.2f}%  (95% CI)")
print(f"  Avg graph density: {test_result_1shot['avg_density']:.3f}")
print(f"  Avg node degree  : {test_result_1shot['avg_degree']:.2f}")

# ── 5-way 5-shot on test split (600 episodes) ─────────────────────────────────
# 5-shot tests whether edge correction loss now contributes (>=2 support/class)
test_sampler_5shot = EpisodicSampler(test_dataset, n_way=5, k_shot=5, n_query=15)
print("\nRunning 5-way 5-shot evaluation on miniImageNet test split (600 episodes)…")
test_result_5shot = evaluate(eval_model, test_sampler_5shot,
                             n_way=5, k_shot=5, n_query=15,
                             n_episodes=600)
print(f"\n  5-way  5-shot: {test_result_5shot['mean_acc']*100:.2f} "
      f"+/- {test_result_5shot['ci_95']*100:.2f}%  (95% CI)")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print(f"{'Method':<30} {'5-way 1-shot':>12}  {'5-way 5-shot':>12}")
print("-" * 55)
print(f"{'DEKAE (ours)':<30} "
      f"{test_result_1shot['mean_acc']*100:>10.2f}%  "
      f"{test_result_5shot['mean_acc']*100:>10.2f}%")
print("=" * 55)
print("Note: FSAKE reported 68.9% (1-shot) on miniImageNet — direct comparison target.")

# ── Save results to Drive ─────────────────────────────────────────────────────
import json
final_results = {
    "dataset"     : ACTIVE_DATASET,
    "1shot_acc"   : test_result_1shot["mean_acc"],
    "1shot_ci95"  : test_result_1shot["ci_95"],
    "5shot_acc"   : test_result_5shot["mean_acc"],
    "5shot_ci95"  : test_result_5shot["ci_95"],
    "avg_density" : test_result_1shot["avg_density"],
}
with open(str(RESULTS_DIR / "dekae_test_results.json"), "w") as f:
    json.dump(final_results, f, indent=2)
print(f"\nResults saved → {RESULTS_DIR}/dekae_test_results.json")


## Section 14: Synthetic Graph Recovery Experiment (Section 2.3)

Planted-partition protocol: noise level $\sigma$ calibrated so k-NN achieves ~65% Graph F1, leaving room for DEKAE to demonstrate genuine topology recovery.

In [ ]:
# ── Section 14: Synthetic Graph Recovery Experiment ──────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

def topology_recovery_metrics(A_pred: torch.Tensor,
                               A_star: torch.Tensor,
                               threshold: float = 0.5) -> dict:
    """
    Compare predicted adjacency against ground-truth planted adjacency.

    threshold : binarisation cutoff.
        - Use 0.5 for DEKAE (sigmoid outputs in (0,1)).
        - Use 1e-9 for k-NN (row-normalised values ~1/k, never > 0.5).
    """
    A_pred_bin = (A_pred > threshold).float()
    # Remove diagonal
    N = A_pred.size(0)
    mask = 1 - torch.eye(N, device=A_pred.device)
    p = A_pred_bin[mask.bool()].cpu().numpy()
    g = A_star[mask.bool()].cpu().numpy()

    precision = precision_score(g, p, zero_division=0)
    recall    = recall_score(g, p, zero_division=0)
    f1        = f1_score(g, p, zero_division=0)
    return {"edge_precision": precision, "edge_recall": recall, "graph_f1": f1}


def knn_recovery_baseline(X: torch.Tensor, A_star: torch.Tensor, k: int):
    """Run static k-NN and measure topology recovery.
    Uses threshold=1e-9 because row-normalised k-NN values are ~1/k << 0.5.
    """
    A_knn = build_knn_adjacency(X, k=k)
    return topology_recovery_metrics(A_knn, A_star, threshold=1e-9)


def synthetic_recovery_experiment(n_way=5, n_per_class=5, feat_dim=64,
                                   sigma_values=(0.3, 0.6, 0.8, 1.0),
                                   n_trials=50, knn_k=5,
                                   trained_model: "DEKAEModel | None" = None):
    """
    Run the topology-recovery experiment across multiple noise levels.

    trained_model : if provided, use the trained model's edge/dynamic modules
                    for DEKAE F1 measurement (post-training fidelity test).
                    When None, uses freshly-initialised modules (pre-training baseline).

    NOTE: The pre-training baseline (trained_model=None) is expected to have
    near-zero F1 — this only establishes the random-init lower bound.
    The meaningful result is the post-training run using the saved checkpoint.

    Returns a pandas DataFrame for easy tabulating.
    """
    try:
        import pandas as pd

        # If trained model is provided, use its embed_dim and first-layer modules
        if trained_model is not None:
            feat_dim = trained_model.embed_dim
            _eim_fixed = trained_model.edge_modules[0]
            _dtm_fixed = trained_model.dynamic_modules[0]
            trained_model.eval()
            print(f"Using trained model modules (feat_dim={feat_dim})")
        else:
            _eim_fixed = None
            _dtm_fixed = None
            print(f"Using fresh (untrained) modules (feat_dim={feat_dim})")

        results = []
        for sigma in sigma_values:
            knn_f1s, ours_f1s = [], []
            for _ in range(n_trials):
                X, y, A_star = synthetic_episode(
                    n_way=n_way, n_nodes_per_class=n_per_class,
                    feat_dim=feat_dim, sigma=sigma)

                # k-NN baseline (threshold=1e-9 — row-normalised values are ~0.2)
                knn_met = knn_recovery_baseline(X, A_star, k=knn_k)
                knn_f1s.append(knn_met["graph_f1"])

                # DEKAE topology recovery
                A_init = build_knn_adjacency(X, k=knn_k)
                with torch.no_grad():
                    if _eim_fixed is not None:
                        # Use trained model's learned modules
                        E_, src_, dst_ = _eim_fixed(X, A_init)
                        A_dek, _       = _dtm_fixed(X, E_, src_, dst_,
                                                     sparsity_reg=False)
                    else:
                        # Fresh random modules (pre-training baseline)
                        eim_tmp = EdgeIncidenceModule(feat_dim, feat_dim).to(X.device)
                        dtm_tmp = DynamicTopologyModule(feat_dim, rank=8,
                                                        edge_dim=feat_dim).to(X.device)
                        E_, src_, dst_ = eim_tmp(X, A_init)
                        A_dek, _       = dtm_tmp(X, E_, src_, dst_,
                                                  sparsity_reg=False)
                ours_met = topology_recovery_metrics(A_dek.detach(), A_star)
                ours_f1s.append(ours_met["graph_f1"])

            results.append({
                "sigma"        : sigma,
                "kNN_F1_mean"  : round(np.mean(knn_f1s), 4),
                "kNN_F1_std"   : round(np.std(knn_f1s), 4),
                "DEKAE_F1_mean": round(np.mean(ours_f1s), 4),
                "DEKAE_F1_std" : round(np.std(ours_f1s), 4),
            })

        df = pd.DataFrame(results)
        print(df.to_string(index=False))
        return df
    except ImportError:
        print("pandas not available — run: !pip install pandas")
        return None


# ── Run 1: pre-training baseline (lower bound — expected near-zero F1) ───────
# This establishes the random-init baseline only. Do NOT interpret this as
# the model's topology recovery capability. Run 2 (below) is the real test.
print("=== Synthetic topology-recovery: untrained model (lower bound) ===")
df_recovery_pretrain = synthetic_recovery_experiment(
    n_way=5, n_per_class=5, feat_dim=64,
    sigma_values=[0.3, 0.6, 0.8, 1.0],
    n_trials=30, knn_k=5,
    trained_model=None,
)

# ── Run 2: post-training (MEANINGFUL result — run AFTER Section 12b) ─────────
# Uses the trained checkpoint loaded in Section 13b.
# If the trained model is not loaded yet, this block is skipped automatically.
# To run manually: set FORCE_RUN2 = True after loading eval_model.
FORCE_RUN2 = False   # ← set True after training; or it auto-runs if eval_model exists

if FORCE_RUN2 or "eval_model" in dir():
    _trained = eval_model if "eval_model" in dir() else None
    if _trained is not None:
        print("\n=== Synthetic topology-recovery: TRAINED model (meaningful result) ===")
        df_recovery_trained = synthetic_recovery_experiment(
            n_way=5, n_per_class=5, feat_dim=128,   # feat_dim matches trained backbone
            sigma_values=[0.3, 0.6, 0.8, 1.0],
            n_trials=30, knn_k=5,
            trained_model=_trained,
        )

        # Compare pre vs post training
        if df_recovery_pretrain is not None and df_recovery_trained is not None:
            import pandas as pd
            df_compare = df_recovery_pretrain[["sigma", "kNN_F1_mean", "DEKAE_F1_mean"]].copy()
            df_compare.columns = ["sigma", "kNN_F1", "DEKAE_untrained_F1"]
            df_compare["DEKAE_trained_F1"] = df_recovery_trained["DEKAE_F1_mean"].values
            print("\n=== Pre vs Post Training Topology Recovery ===")
            print(df_compare.to_string(index=False))
            df_compare.to_csv(str(RESULTS_DIR / "topology_recovery_comparison.csv"), index=False)
            print(f"\nSaved → {RESULTS_DIR}/topology_recovery_comparison.csv")
    else:
        print("eval_model not found. Load checkpoint in Section 13b first, then re-run.")
else:
    print("\n  Run 2 (trained model) deferred. Load checkpoint in Section 13b and "
          "set FORCE_RUN2=True to execute.")


## Section 15: Ablation Study Runner

Each group (A–G from the plan) can be run by calling `run_ablation(cfg_dict)`. Configs are defined per variant and results collected into a comparison table.

In [ ]:
# -- Section 15: Ablation Study Runner -----------------------------------------
import copy

# -- Base config (derived from CFG) --------------------------------------------
# IMPORTANT: extended from 30->100 epochs and 100->300 ep/epoch so ablation
# results are comparable to the full training run (30 epochs was too short
# to rank variants reliably -- all differences were within CI).
BASE = copy.deepcopy(CFG)
BASE["n_epochs"]          = 100
BASE["n_episodes_train"]  = 300
BASE["n_eval_episodes"]   = 200

# -------------------------------------------------------------------------------
# Group A: Topology Dynamics  (plan §5.1)
#
# Ablation table — progressive addition of DEKAE's novelties:
#
#   A0  ProtoNet         n_gnn_layers=0: pure backbone, no graph propagation
#   A1  HGNN             static k-NN adjacency + plain A@H message passing
#   A2  FSAKE            dynamic scalar MLP adjacency A^(l)=softmax(MLP(|h_i-h_j|)),
#                        plain A'@H — approximated via use_dynamic=True, use_edge_proj=False
#   A3  static+edge      static k-NN adjacency + Case 1 incidence edge gating
#                        (shows edge features improve even a fixed topology)
#   A4  dyn+Case1        dynamic adjacency + Case 1 edge-gated node aggr.
#                        (X' = σ(g(E)) · A' @ H)
#   A5  dyn+Case2        dynamic adjacency + Case 2 edge-vector node update
#                        (X' = A_edge @ E @ W') — the core DEKAE novelty
#   A6  DEKAE_noLMT      A5 + supervised edge loss (λ_edge=0.5), no LMT
#   A7  DEKAE_full       A6 + Latent Mediator Transformer
#
# Key comparisons (each isolates ONE variable):
#   A1 − A0: does GNN over static k-NN help at all?
#   A2 − A1: dynamic topology (FSAKE scalar) vs. static k-NN (HGNN)
#   A3 − A1: do incidence-based edge features improve a static topology?
#   A4 − A2: do edge features improve a dynamic topology? (Case 1 node update)
#   A4 − A3: does dynamic topology improve over static when using edge features?
#   A5 − A4: Case 2 (vector edge update) vs Case 1 (scalar gated aggr.)
#   A6 − A5: value of supervised edge-level loss
#   A7 − A6: value of LMT cross-set refinement
#
# All A0–A6 variants use use_lmt=False so the GNN contribution is cleanly
# isolated. Group H tests the LMT module independently (H1a vs H1b = A6 vs A7).
# -------------------------------------------------------------------------------
ABLATION_CONFIGS = {

    # A0 -- ProtoNet anchor: pure backbone, zero graph propagation
    # n_gnn_layers=0 skips the GNN loop entirely — classification on raw embeddings
    "A0_ProtoNet": {**BASE,
        "n_gnn_layers": 0,
        "use_dynamic": False, "use_edge_proj": False,
        "use_case2": False, "use_lmt": False,
        "lambda_edge": 0.0, "sparsity_mode": "none"},

    # A1 -- HGNN baseline: static k-NN adjacency + plain A@H message passing
    # (The FSAKE predecessor; static adjacency, no edge features, no edge loss)
    "A1_HGNN": {**BASE,
        "use_dynamic": False, "use_edge_proj": False,
        "use_case2": False,   "use_lmt": False,
        "lambda_edge": 0.0, "sparsity_mode": "none"},

    # A2 -- FSAKE-style: dynamic scalar adjacency, plain A'@H node update
    # Approximates FSAKE's per-layer A^(l) = softmax(MLP(|h_i-h_j|)) regime.
    # use_edge_proj=False → node update stays as A'@H (no Case 1/2).
    "A2_FSAKE": {**BASE,
        "use_dynamic": True, "use_edge_proj": False,
        "use_case2": False,  "use_lmt": False,
        "lambda_edge": 0.0, "sparsity_mode": "none"},

    # A3 -- Static k-NN adjacency + Case 1 incidence edge gating
    # Shows that even a FIXED topology benefits from DEKAE's incidence edge features
    # (isolates the edge-feature value from the dynamic-topology value).
    # Distinct from A1 (adds edge proj) and A4 (keeps static, not dynamic).
    "A3_static_edge": {**BASE,
        "use_dynamic": False, "use_edge_proj": True,
        "use_case2": False,   "use_lmt": False,
        "lambda_edge": 0.0},

    # A4 -- Dynamic adjacency + Case 1 node update (edge-gated aggregation)
    # X' = σ(g(E)) · A' @ H  — dynamic topology + scalar edge gating
    "A4_dyn_case1": {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": False,  "use_lmt": False,
        "lambda_edge": 0.0},

    # A5 -- Dynamic adjacency + Case 2 node update (vector edge features)
    # X' = A_edge @ E @ W'  — the stronger DEKAE novelty
    "A5_dyn_case2": {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": False,
        "lambda_edge": 0.0},

    # A6 -- Full DEKAE GNN: dynamic + Case 2 + supervised edge loss, no LMT
    "A6_DEKAE_noLMT": {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": False,
        "lambda_edge": 0.5},

    # A7 -- Full DEKAE with LMT (complete proposed model)
    "A7_DEKAE_full": {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": True,
        "lambda_edge": 0.5},

    # -- Group B: Selection Strategy (plan §5.2) ---------------------------------
    "B1_max_k5":     {**BASE, "use_dynamic": False, "use_lmt": False,
                      "topk_k": 5, "lambda_edge": 0.0,
                      "selection_strategy": "max"},
    "B2_nearest_k5": {**BASE, "use_dynamic": False, "use_lmt": False,
                      "topk_k": 5, "lambda_edge": 0.0,
                      "selection_strategy": "nearest_centroid"},
    "B3_dynamic":    {**BASE, "use_dynamic": True,  "use_lmt": False,
                      "lambda_edge": 0.5, "selection_strategy": "max"},

    # -- Group G2: N-way / K-shot Config Sensitivity (plan §5.7 G2) --------------
    "G2_5w1s":   {**BASE, "n_way": 5,  "k_shot": 1, "n_query": 15,
                  "use_lmt": False, "use_dynamic": True, "lambda_edge": 0.5},
    "G2_5w5s":   {**BASE, "n_way": 5,  "k_shot": 5, "n_query": 15,
                  "use_lmt": False, "use_dynamic": True, "lambda_edge": 0.5},
    "G2_10w1s":  {**BASE, "n_way": 10, "k_shot": 1, "n_query": 15,
                  "use_lmt": False, "use_dynamic": True, "lambda_edge": 0.5},
    "G2_10w5s":  {**BASE, "n_way": 10, "k_shot": 5, "n_query": 15,
                  "use_lmt": False, "use_dynamic": True, "lambda_edge": 0.5},

    # -- Group G3: Backbone / Embed-Dim Sensitivity (plan §5.7 G3) ---------------
    "G3_embed64":  {**BASE, "embed_dim": 64,  "use_lmt": False,
                    "use_dynamic": True, "lambda_edge": 0.5},
    "G3_embed128": {**BASE, "embed_dim": 128, "use_lmt": False,
                    "use_dynamic": True, "lambda_edge": 0.5},  # default
    "G3_embed256": {**BASE, "embed_dim": 256, "use_lmt": False,
                    "use_dynamic": True, "lambda_edge": 0.5},

    # -- Group C: Edge Feature Projection Type (plan §5.3) -----------------------
    "C1_no_edge_feat":  {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": False,
        "lambda_edge": 0.5,  "edge_proj_type": "none"},
    "C2_linear_proj":   {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": False,
        "lambda_edge": 0.5,  "edge_proj_type": "linear"},
    "C3_mlp_proj":      {**BASE,
        "use_dynamic": True, "use_edge_proj": True,
        "use_case2": True,   "use_lmt": False,
        "lambda_edge": 0.5,  "edge_proj_type": "mlp"},

    # -- Group E: Sparsity -------------------------------------------------------
    # WARNING: E4_laplacian collapsed to ~26% accuracy in initial runs.
    "E1_no_sparse":   {**BASE, "sparsity_mode": "none",     "use_lmt": False},
    "E2_l1":          {**BASE, "sparsity_mode": "l1",       "use_lmt": False},
    "E3_topk":        {**BASE, "sparsity_mode": "topk",     "use_lmt": False},
    "E4_laplacian":   {**BASE, "sparsity_mode": "laplacian","use_lmt": False},  # known-harmful

    # -- Group F: Edge MLP Capacity ----------------------------------------------
    "F1_full_rank64": {**BASE, "rank": 64,  "use_lmt": False},
    "F2_lowrank16":   {**BASE, "rank": 16,  "use_lmt": False},
    "F3_dotprod1":    {**BASE, "rank": 1,   "use_lmt": False},

    # -- Group D: Seed Stability (plan §5.4) -------------------------------------
    "D_seed42":  {**BASE, "use_dynamic": True, "use_edge_proj": True,
                  "use_case2": True, "use_lmt": False, "lambda_edge": 0.5,
                  "seed": 42},
    "D_seed0":   {**BASE, "use_dynamic": True, "use_edge_proj": True,
                  "use_case2": True, "use_lmt": False, "lambda_edge": 0.5,
                  "seed": 0},
    "D_seed1":   {**BASE, "use_dynamic": True, "use_edge_proj": True,
                  "use_case2": True, "use_lmt": False, "lambda_edge": 0.5,
                  "seed": 1},
    "D_seed7":   {**BASE, "use_dynamic": True, "use_edge_proj": True,
                  "use_case2": True, "use_lmt": False, "lambda_edge": 0.5,
                  "seed": 7},
    "D_seed123": {**BASE, "use_dynamic": True, "use_edge_proj": True,
                  "use_case2": True, "use_lmt": False, "lambda_edge": 0.5,
                  "seed": 123},

    # -- Group G5: Lambda_edge Sensitivity (plan §5.7 G5) -----------------------
    "G5_lam0":   {**BASE, "lambda_edge": 0.0, "use_lmt": False},
    "G5_lam01":  {**BASE, "lambda_edge": 0.1, "use_lmt": False},
    "G5_lam05":  {**BASE, "lambda_edge": 0.5, "use_lmt": False},
    "G5_lam1":   {**BASE, "lambda_edge": 1.0, "use_lmt": False},
    "G5_lam2":   {**BASE, "lambda_edge": 2.0, "use_lmt": False},

    # -- Group H: Latent Mediator Transformer Ablation (plan §5.8) ---------------
    # H2: Number of mediator tokens m
    "H2_m2":     {**BASE, "use_lmt": True, "n_mediators": 2,   "lambda_edge": 0.5},
    "H2_m8":     {**BASE, "use_lmt": True, "n_mediators": 8,   "lambda_edge": 0.5},
    "H2_m16":    {**BASE, "use_lmt": True, "n_mediators": 16,  "lambda_edge": 0.5},
    "H2_m32":    {**BASE, "use_lmt": True, "n_mediators": 32,  "lambda_edge": 0.5},
    # H3: LMT depth
    "H3_L1":     {**BASE, "use_lmt": True, "lmt_layers": 1, "lambda_edge": 0.5},
    "H3_L3":     {**BASE, "use_lmt": True, "lmt_layers": 3, "lambda_edge": 0.5},
    "H3_L5":     {**BASE, "use_lmt": True, "lmt_layers": 5, "lambda_edge": 0.5},
    # H4: Mediator Initialisation Strategy
    "H4_learned": {**BASE, "use_lmt": True, "lmt_init_strategy": "learned",
                   "lambda_edge": 0.5},
    "H4_fixed":   {**BASE, "use_lmt": True, "lmt_init_strategy": "fixed",
                   "lambda_edge": 0.5},
    "H4_zero":    {**BASE, "use_lmt": True, "lmt_init_strategy": "zero",
                   "lambda_edge": 0.5},
    # H5: LMT Phase Ablation
    "H5a_gather_only":     {**BASE, "use_lmt": True, "lmt_phases": "gather_only",
                             "lambda_edge": 0.5},
    "H5b_distribute_only": {**BASE, "use_lmt": True, "lmt_phases": "distribute_only",
                             "lambda_edge": 0.5},
    "H5c_both":            {**BASE, "use_lmt": True, "lmt_phases": "both",
                             "lambda_edge": 0.5},

    # -- Group I: Multi-Level Edge Supervision (plan §5.9) -----------------------
    "I1_penultimate":  {**BASE, "use_lmt": True, "edge_loss_mode": "penultimate",
                        "lambda_edge": 0.5},
    "I2_all_equal":    {**BASE, "use_lmt": True, "edge_loss_mode": "all",
                        "lambda_edge": 0.5},
    "I3_all_decaying": {**BASE, "use_lmt": True, "edge_loss_mode": "decaying",
                        "lambda_edge": 0.5},
}


# -- Ablation result cache helpers (Drive resumability) -------------------------
_ABLATION_CACHE_FILE = None   # resolved lazily after RESULTS_DIR is available

def _ablation_cache_path():
    """Return the path for the shared ablation results JSON cache on Drive."""
    global _ABLATION_CACHE_FILE
    if _ABLATION_CACHE_FILE is None:
        _ABLATION_CACHE_FILE = RESULTS_DIR / "ablation_results.json"
    return _ABLATION_CACHE_FILE

def _load_ablation_cache() -> dict:
    """Load {name: result_dict} from the Drive cache file, or return {}."""
    p = _ablation_cache_path()
    if p.exists():
        try:
            with open(str(p)) as f:
                data = json.load(f)
            if isinstance(data, list):
                return {r["name"]: r for r in data if "name" in r}
            if isinstance(data, dict):
                return data
        except Exception:
            pass
    return {}

def _save_ablation_cache(cache: dict):
    """Persist the ablation cache dict to Drive as a list-of-dicts."""
    p = _ablation_cache_path()
    try:
        with open(str(p), "w") as f:
            json.dump(list(cache.values()), f, indent=2)
    except Exception as _e:
        print(f"  WARNING: Could not write ablation cache: {_e}")


def run_ablation(name: str, cfg: dict, episode_fn_train, episode_fn_val,
                 log_wandb: bool = False):
    """
    Train and evaluate a single ablation variant.
    Applies ALL ablation flags from cfg via build_model() and the train()
    warm-up logic — no manual set_use_*() calls needed here.
    Returns a summary dict with name, mean_acc, ci_95.

    Resume behaviour
    ----------------
    If RESULTS_DIR/ablation_results.json already contains a completed entry
    for `name`, the variant is skipped and the cached result is returned
    immediately — no re-training needed after a runtime reset.
    """
    # -- Skip if already completed ---------------------------------------------
    _cache = _load_ablation_cache()
    if name in _cache:
        cached = _cache[name]
        print(f"\n[SKIP] '{name}' already in Drive cache: "
              f"{cached.get('mean_acc', 0)*100:.2f}% ± {cached.get('ci_95', 0)*100:.2f}%")
        return cached

    print(f"\n{'='*60}")
    print(f"Running ablation: {name}")
    print(f"{'='*60}")
    set_seed(cfg.get("seed", 42))

    # build_model() passes all flags (use_dynamic, use_edge_proj, use_case2,
    # use_lmt) to DEKAEModel; train() additionally respects cfg["use_dynamic"]
    # for the warm-up epoch logic. No explicit set_use_*() calls needed.
    trained_model, history = train(
        cfg, episode_fn_train, episode_fn_val,
        run_name=name, log_wandb=log_wandb
    )

    # Evaluate
    res = evaluate(trained_model, episode_fn_val,
                   cfg["n_way"], cfg["k_shot"], cfg["n_query"],
                   n_episodes=cfg.get("n_eval_episodes", 200))
    res["name"] = name
    print(f"-> {name}: {res['mean_acc']*100:.2f} +/- {res['ci_95']*100:.2f}%")

    # -- Persist result to Drive cache -----------------------------------------
    _cache[name] = res
    _save_ablation_cache(_cache)
    print(f"  [SAVED] Ablation result cached -> {_ablation_cache_path().name}")

    return res


def run_all_ablations(groups: list, episode_fn_train, episode_fn_val):
    """Run a subset of ablation groups and collect results.
    Already-completed variants (present in the Drive cache) are skipped."""
    results = []
    for name in groups:
        if name not in ABLATION_CONFIGS:
            print(f"WARNING: Unknown ablation key: {name}")
            continue
        res = run_ablation(name, ABLATION_CONFIGS[name],
                           episode_fn_train, episode_fn_val)
        results.append(res)
    return results


print("Ablation configs defined.")
print("Group A variants :", [k for k in ABLATION_CONFIGS if k.startswith("A")])
print("Group B variants :", [k for k in ABLATION_CONFIGS if k.startswith("B")])
print("Group C variants :", [k for k in ABLATION_CONFIGS if k.startswith("C")])
print("Group D variants :", [k for k in ABLATION_CONFIGS if k.startswith("D")])
print("Group E variants :", [k for k in ABLATION_CONFIGS if k.startswith("E")])
print("Group G2 variants:", [k for k in ABLATION_CONFIGS if k.startswith("G2")])
print("Group G3 variants:", [k for k in ABLATION_CONFIGS if k.startswith("G3")])
print("Group G5 variants:", [k for k in ABLATION_CONFIGS if k.startswith("G5")])
print("Group H variants :", [k for k in ABLATION_CONFIGS if k.startswith("H")])

In [ ]:
# ── Section 15b: Execute Ablation Groups A, E, and H ─────────────────────────
# Run AFTER Section 12b training completes.
# Each variant trains for 100 epochs × 300 episodes ≈ 30,000 episodes total.
#
# Estimated time per variant on T4: ~60–90 min
# Total for Groups A+E core (8 variants, E4 excluded): ~8–10 hours on T4.
# Recommended: run one group at a time, saving results between sessions.
#
# Group A proves each novelty component adds value (plan §5.1):
#   A0→A1: GNN propagation helps over pure backbone
#   A1→A2: dynamic topology (FSAKE) > static k-NN (HGNN)
#   A2→A4: incidence edge features further improve dynamic topology
#   A4→A5: Case 2 vector edge update > Case 1 scalar gating
#   A5→A6: supervised edge-level loss adds value
#   A6→A7: LMT cross-set refinement adds value  [Group H]
# Group E proves: sparsity prevents collapse; E4_laplacian is excluded.
# Group H proves: LMT adds value on top of A6_DEKAE_noLMT.

import pandas as pd

def print_ablation_table(results: list, title: str = "Ablation Results"):
    print(f"\n{'='*65}")
    print(f"  {title}")
    print(f"{'='*65}")
    print(f"{'Variant':<22} {'Acc (%)':>9}  {'CI95':>6}  {'Density':>8}  {'Intra-E':>8}")
    print("-" * 65)
    for r in sorted(results, key=lambda x: -x["mean_acc"]):
        print(f"  {r['name']:<20} {r['mean_acc']*100:>8.2f}% "
              f"±{r['ci_95']*100:>5.2f}%  "
              f"{r.get('avg_density', 0):>7.3f}  "
              f"{r.get('avg_intra_ratio', 0):>8.3f}")
    print("=" * 65)


# ── Group A: Topology Dynamics ────────────────────────────────────────────────
# Maps to plan §5.1 ablation table (A0–A7).
# A0  ProtoNet        n_gnn_layers=0, pure backbone (no graph at all)
# A1  HGNN            static k-NN adjacency, plain A@H message passing
# A2  FSAKE           dynamic scalar adjacency, plain A'@H (FSAKE baseline)
# A3  static+edge     static k-NN + Case 1 incidence edge gating (isolates edge value)
# A4  dyn+Case1       dynamic adjacency + Case 1 scalar edge gating
# A5  dyn+Case2       dynamic adjacency + Case 2 vector edge update (core DEKAE)
# A6  DEKAE_noLMT     A5 + supervised edge loss  [no LMT]
# All variants use use_lmt=False so GNN contribution is cleanly isolated.
print("Running Group A ablations (topology dynamics)…")
group_a_results = run_all_ablations(
    groups            = ["A0_ProtoNet", "A1_HGNN", "A2_FSAKE",
                         "A3_static_edge", "A4_dyn_case1",
                         "A5_dyn_case2", "A6_DEKAE_noLMT"],
    episode_fn_train  = train_sampler_aug,
    episode_fn_val    = val_sampler,
)
print_ablation_table(group_a_results, "Group A — Topology Dynamics (no LMT)")

# ── Group E: Sparsity Regularisation (E4_laplacian run separately) ───────────
# ⚠ E4_laplacian collapsed to 26.24% in prior run (near random chance).
# Run it last so it doesn't waste compute time before seeing the main results.
print("\nRunning Group E core ablations (sparsity, E4 excluded)…")
group_e_core_results = run_all_ablations(
    groups            = ["E1_no_sparse", "E2_l1", "E3_topk"],
    episode_fn_train  = train_sampler_aug,
    episode_fn_val    = val_sampler,
)
print_ablation_table(group_e_core_results, "Group E — Sparsity Regularisation (core)")

# ── E4_laplacian (run separately, known to be harmful) ───────────────────────
RUN_E4 = False   # ← set True only if you want the negative-ablation data point
if RUN_E4:
    print("\n⚠ Running E4_laplacian — expected to collapse to ~26% (negative ablation)…")
    group_e4_result = run_all_ablations(
        groups            = ["E4_laplacian"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    group_e_core_results += group_e4_result
    print_ablation_table(group_e_core_results, "Group E — Sparsity Regularisation (all)")
else:
    print("\n  (E4_laplacian skipped — set RUN_E4=True to include as negative ablation)")

# ── Group H: LMT Ablation (plan §5.8) ────────────────────────────────────────
# H1a = A6_DEKAE_noLMT (baseline, already in Group A above)
# H1b = A7_DEKAE_full  (with LMT)
# H2 = mediator token count sensitivity
# H3 = LMT depth sensitivity
RUN_GROUP_H = False   # ← set True to run LMT ablations (each ~60–90 min)
if RUN_GROUP_H:
    print("\nRunning Group H ablations (LMT module)…")
    # H1: with vs without LMT
    group_h1_results = run_all_ablations(
        groups            = ["A6_DEKAE_noLMT", "A7_DEKAE_full"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    print_ablation_table(group_h1_results, "Group H1 — LMT vs No-LMT")

    # H2: mediator count sensitivity
    group_h2_results = run_all_ablations(
        groups            = ["H2_m2", "H2_m8", "H2_m16", "H2_m32"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    print_ablation_table(group_h2_results, "Group H2 — Mediator Token Count (m)")

    # H3: LMT depth sensitivity
    group_h3_results = run_all_ablations(
        groups            = ["H3_L1", "H3_L3", "H3_L5"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    print_ablation_table(group_h3_results, "Group H3 — LMT Depth (L_lmt)")
else:
    print("\n  (Group H skipped — set RUN_GROUP_H=True after Group A training completes)")

# ── Group H4: Mediator Initialisation Ablation (plan §5.8 H4) ────────────────
RUN_GROUP_H4 = False   # ← set True to test mediator init strategies
if RUN_GROUP_H4:
    print("\nRunning Group H4 ablations (mediator token init strategy)…")
    group_h4_results = run_all_ablations(
        groups            = ["H4_learned", "H4_fixed", "H4_zero"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    print_ablation_table(group_h4_results, "Group H4 — Mediator Init Strategy")
else:
    print("  (Group H4 skipped — set RUN_GROUP_H4=True)")

# ── Group H5: LMT Phase Ablation (plan §5.8 H5) ──────────────────────────────
RUN_GROUP_H5 = False   # ← set True to test gather / distribute phases
if RUN_GROUP_H5:
    print("\nRunning Group H5 ablations (LMT phase activation)…")
    group_h5_results = run_all_ablations(
        groups            = ["H5a_gather_only", "H5b_distribute_only", "H5c_both"],
        episode_fn_train  = train_sampler_aug,
        episode_fn_val    = val_sampler,
    )
    print_ablation_table(group_h5_results, "Group H5 — LMT Phase Ablation")
else:
    print("  (Group H5 skipped — set RUN_GROUP_H5=True)")

# ── Group I: Multi-Level Edge Supervision (plan §5.9) ────────────────────────
RUN_GROUP_I = False   # ← set True to test edge-loss application layer
if RUN_GROUP_I:
    print("\nRunning Group I ablations (multi-level edge supervision)…")
    group_i_results = run_all_ablations(

## Section 15c: Ablation Group C — Edge Feature Projection Type

Compares three levels of edge feature richness (plan §5.3):

| Variant | Edge Projection | Description |
|---------|----------------|-------------|
| **C1** | `none` | No learned edge features — zeros passed to `DynamicTopologyModule` and `EdgeToNodeProjection`; degenerates to adjacency-only MP |
| **C2** | `linear` | Single `nn.Linear(2d → d)` — low-capacity linear projection, no non-linearity |
| **C3** | `mlp` | 2-layer MLP with ReLU (current default) — full expressive edge representation |

**What this proves**: The gain from DEKAE's edge-feature design comes from
non-linear relational representation, not just from having more parameters.
If C3 > C2 > C1, the full MLP is justified. If C2 ≈ C3, a linear projection
suffices and the MLP can be simplified without accuracy loss.

All Group C variants use `use_case2=True`, `use_dynamic=True`, `use_lmt=False`
so the edge-feature contribution to the GNN is cleanly isolated.
The `edge_proj_type` flag added to `EdgeIncidenceModule` and `DEKAEModel`
controls this ablation; `build_model()` reads it from the config dict.

In [ ]:
# ── Section 15c: Group C — Edge Feature Projection Type ──────────────────────
# Run this cell independently to execute only Group C ablations.
# Requires a trained full model (Section 12b) so relative performance is
# measured against a meaningful representation, not random features.
#
# Each variant uses the BASE ablation config (100 epochs × 300 ep/epoch).
# Estimated time: 3 variants × ~60–90 min ≈ 3–4.5 hours on T4.
#
# ── Quick Model Sanity Check (runs immediately, no training needed) ─────────
print('Group C model sanity checks (untrained):')
print('-' * 55)
_s_c = torch.randn(5, 3, 84, 84).to(DEVICE)
_l_c = torch.arange(5).to(DEVICE)
_q_c = torch.randn(15, 3, 84, 84).to(DEVICE)

for _name_c, _ept in [('C1_no_edge_feat', 'none'),
                       ('C2_linear_proj',  'linear'),
                       ('C3_mlp_proj',     'mlp')]:
    _mc = build_model({**BASE, 'edge_proj_type': _ept,
                       'use_dynamic': True, 'use_edge_proj': True,
                       'use_case2': True, 'use_lmt': False,
                       'lambda_edge': 0.5})
    _mc.eval()
    with torch.no_grad():
        _lg_c, _al_c, _mt_c = _mc(_s_c, _l_c, _q_c)
    _np_c = sum(p.numel() for p in _mc.parameters())
    print(f'  {_name_c:<22} edge_proj={_ept:<7}  '
          f'logits={list(_lg_c.shape)}  '
          f'density={_mt_c["graph_density"]:.3f}  '
          f'params={_np_c:,}')

print()
print('All three variants produce valid logits. '
      'Set RUN_GROUP_C=True in Section 15b to run full training.')


## Section 15d: Ablation Group D — Seed Stability & Topology Init Sensitivity

**Group D** (plan §5.4 + Risk 8) validates reproducibility along two axes:

### D1 — Across-Run Seed Stability
Trains the full DEKAE model (same config as `A6_DEKAE_noLMT`) with 5 different
global random seeds `{42, 0, 1, 7, 123}`. Reports:
- Per-seed test accuracy, graph density, and avg degree
- **Mean ± std** and **95% CI** across seeds
- Low std (< 0.5%) → model is robust to initialization; high std → unstable

| Seed | Accuracy | Density | Avg Degree |
|------|----------|---------|------------|
| 42   | (report) | (report)| (report)   |
| 0    | (report) | (report)| (report)   |
| 1    | (report) | (report)| (report)   |
| 7    | (report) | (report)| (report)   |
| 123  | (report) | (report)| (report)   |
| **Mean±std** | (report) | (report)| (report) |

### D2 — Topology Initialization Sensitivity (Risk 8 mitigation)
Fixes the backbone weights (trained checkpoint) and re-initializes *only* the
edge scoring MLP (`DynamicTopologyModule.W1`, `W2`, `edge_scorer`) with
`n_inits=5` different seeds. For each init, runs `n_episodes` episodes and
records the recovered adjacency $A'$. Reports the **mean Frobenius norm**
$\\|A'_{\\text{init}_i} - A'_{\\text{init}_j}\\|_F$ across pairs:
- Low variance → near-unique convergence (validates Risk 8 identifiability claim)
- High variance → topology is sensitive to MLP init (add this as a limitation)


In [ ]:
# -- Section 15d: Group D -- Seed Stability & Topology Init Sensitivity ----
import copy, itertools

# ---------------------------------------------------------------------------
# D1: run_seed_stability()
#   Trains the full model with multiple global seeds.
#   Returns a summary dict and a list of per-seed result dicts.
# ---------------------------------------------------------------------------
def run_seed_stability(cfg,
                       seeds,
                       episode_fn_train,
                       episode_fn_val,
                       episode_fn_test=None,
                       n_test_episodes=600):
    """
    Train the DEKAE model once per seed and collect accuracy + topology metrics.

    Parameters
    ----------
    cfg              : base config dict (seed key will be overridden per run)
    seeds            : list of integer seeds to sweep
    episode_fn_train : callable for training episodes
    episode_fn_val   : callable for validation episodes
    episode_fn_test  : callable for test episodes (optional; uses val if None)
    n_test_episodes  : number of episodes for final accuracy estimate per seed

    Returns
    -------
    summary : dict with mean/std/ci_95 across seeds for accuracy + graph metrics
    results : list of per-seed dicts
    """
    test_fn = episode_fn_test if episode_fn_test is not None else episode_fn_val
    per_seed = []

    for seed in seeds:
        seed_cfg = copy.deepcopy(cfg)
        seed_cfg['seed'] = seed
        run_name = f'D_seed{seed}'
        print(f'\n{"="*55}')
        print(f'  Group D -- seed={seed}  ({seeds.index(seed)+1}/{len(seeds)})')
        print(f'{"="*55}')

        trained, history = train(
            seed_cfg, episode_fn_train, episode_fn_val,
            run_name=run_name, log_wandb=False
        )
        res = evaluate(trained, test_fn,
                       seed_cfg['n_way'], seed_cfg['k_shot'], seed_cfg['n_query'],
                       n_episodes=n_test_episodes)

        # Topology stability: std of graph density across test episodes
        topo_std = float(np.std(res.get('per_episode_density',
                                        [res['avg_density']] * n_test_episodes)))

        per_seed.append({
            'seed'        : seed,
            'acc'         : res['mean_acc'],
            'ci_95'       : res['ci_95'],
            'avg_density' : res['avg_density'],
            'avg_degree'  : res['avg_degree'],
            'topo_std'    : topo_std,
        })
        print(f'  Seed {seed}: acc={res["mean_acc"]*100:.2f}%  '
              f'density={res["avg_density"]:.3f}  topo_std={topo_std:.4f}')

    accs      = [r['acc']         for r in per_seed]
    densities = [r['avg_density'] for r in per_seed]

    # Run D2 topology-init sensitivity using the last trained model's backbone
    frob_mean, frob_std = topology_init_sensitivity(
        backbone    = trained.backbone,
        edge_module = trained.edge_modules[0],
        dyn_module  = trained.dynamic_modules[0],
        episode_fn  = test_fn,
        n_episodes  = 30,
        n_inits     = 5,
        knn_k       = cfg['topk_k'],
        embed_dim   = cfg['embed_dim'],
        rank        = cfg['rank'],
    )

    summary = {
        'seeds'              : seeds,
        'mean_acc'           : float(np.mean(accs)),
        'std_acc'            : float(np.std(accs)),
        'ci_95'              : 1.96 * float(np.std(accs)) / (len(accs) ** 0.5),
        'mean_density'       : float(np.mean(densities)),
        'std_density'        : float(np.std(densities)),
        'topo_init_frob_mean': frob_mean,
        'topo_init_frob_std' : frob_std,
    }

    # Print summary table
    print(f'\n{"="*65}')
    print(f'  Group D -- Seed Stability Summary ({len(seeds)} seeds)')
    print(f'{"="*65}')
    print(f'  {"Seed":<8} {"Acc (%)":>9}  {"CI95":>6}  {"Density":>8}  {"Avg Deg":>8}')
    print(f'  {"-"*50}')
    for r in per_seed:
        print(f'  {r["seed"]:<8} {r["acc"]*100:>8.2f}%  '
              f'+-{r["ci_95"]*100:>4.2f}%  '
              f'{r["avg_density"]:>8.3f}  {r["avg_degree"]:>8.2f}')
    print(f'  {"-"*50}')
    print(f'  {"Mean+-std":<8} {summary["mean_acc"]*100:>8.2f}%  '
          f'+-{summary["std_acc"]*100:>4.2f}%  '
          f'{summary["mean_density"]:>8.3f}')
    print(f'  {"95% CI":<8} +-{summary["ci_95"]*100:.2f}%')
    print(f'\n  Topology Init Sensitivity (D2):')
    print(f'    Mean Frobenius norm across init pairs: '
          f'{summary["topo_init_frob_mean"]:.4f} +- {summary["topo_init_frob_std"]:.4f}')
    print(f'    (Near zero = near-unique topology convergence; validates Risk 8)')
    print(f'{"="*65}')

    # Save to Drive
    with open(str(RESULTS_DIR / 'ablation_group_d.json'), 'w') as _f:
        json.dump({'summary': summary, 'per_seed': per_seed}, _f, indent=2)
    print(f'\nGroup D results saved -> {RESULTS_DIR}/ablation_group_d.json')

    return summary, per_seed


# ---------------------------------------------------------------------------
# D2: topology_init_sensitivity()
#   Fixes the backbone; re-inits only edge MLP weights; measures Frobenius
#   norm between recovered adjacency matrices across init seeds.
#   Low norm variance -> near-unique convergence (plan section 1.5 Risk 8).
# ---------------------------------------------------------------------------
def topology_init_sensitivity(backbone, edge_module, dyn_module,
                               episode_fn, n_episodes=30,
                               n_inits=5, knn_k=5,
                               embed_dim=128, rank=16):
    """
    For each of n_inits random initializations of the edge scoring weights,
    run n_episodes episodes using the FIXED backbone and collect the recovered
    adjacency A_prime. Report pairwise Frobenius norms between adjacency matrices.

    The backbone weights are kept frozen; only DynamicTopologyModule and
    EdgeIncidenceModule weights are re-initialized per run. This isolates
    topology sensitivity from representation sensitivity.

    Returns
    -------
    frob_mean : float  mean pairwise Frobenius norm across init seed pairs
    frob_std  : float  std of pairwise Frobenius norms
    """
    backbone.eval()
    backbone.to(DEVICE)

    adj_lists = []   # one mean adjacency per init

    for init_seed in range(n_inits):
        torch.manual_seed(init_seed)
        # Fresh copies with new random weights
        eim_fresh = EdgeIncidenceModule(embed_dim, embed_dim, hidden=64).to(DEVICE)
        dtm_fresh = DynamicTopologyModule(embed_dim, rank, embed_dim).to(DEVICE)

        # Explicit re-init to ensure independence from global state
        for mod in [eim_fresh, dtm_fresh]:
            for p in mod.parameters():
                if p.dim() >= 2:
                    nn.init.xavier_uniform_(p)
                else:
                    nn.init.zeros_(p)

        eim_fresh.eval(); dtm_fresh.eval()
        ep_adjs = []

        with torch.no_grad():
            for _ in range(n_episodes):
                s_imgs, s_lbl, q_imgs, q_lbl = episode_fn(5, 1, 15)
                all_imgs = torch.cat([s_imgs, q_imgs], dim=0).to(DEVICE)
                H = backbone(all_imgs)                        # fixed backbone
                A_init = build_knn_adjacency(H, k=knn_k)
                E_f, src_f, dst_f = eim_fresh(H, A_init)
                A_prime, _ = dtm_fresh(H, E_f, src_f, dst_f, sparsity_reg=False)
                ep_adjs.append(A_prime.cpu())

        # Mean adjacency over episodes (reduces episode noise)
        mean_adj = torch.stack(ep_adjs).mean(dim=0)
        adj_lists.append(mean_adj)

    # Pairwise Frobenius norms
    frob_norms = []
    for i, j in itertools.combinations(range(n_inits), 2):
        frob = (adj_lists[i] - adj_lists[j]).norm(p='fro').item()
        frob_norms.append(frob)

    frob_mean = float(np.mean(frob_norms)) if frob_norms else 0.0
    frob_std  = float(np.std(frob_norms))  if frob_norms else 0.0

    print(f'  D2 Topology Init Sensitivity: {n_inits} inits x {n_episodes} episodes')
    print(f'    Pairwise Frobenius norms: {[round(f,4) for f in frob_norms]}')
    print(f'    Mean={frob_mean:.4f}  Std={frob_std:.4f}')
    if frob_mean < 0.05:
        print('    -> Low variance: topology converges near-uniquely (Risk 8 satisfied)')
    else:
        print('    -> Moderate/high variance: topology sensitive to MLP init')

    return frob_mean, frob_std


# ---------------------------------------------------------------------------
# Quick sanity check (runs without training)
# ---------------------------------------------------------------------------
print('Group D functions defined:')
print('  run_seed_stability()        -- D1: trains full model N seeds, reports mean+-std+-CI')
print('  topology_init_sensitivity() -- D2: fixes backbone, varies MLP init, Frobenius norm')
print()
print('Quick D2 sanity check (untrained backbone)...')
_bb_dummy  = Conv4(embed_dim=128).to(DEVICE)
_eim_dummy = EdgeIncidenceModule(128, 128, hidden=64).to(DEVICE)
_dtm_dummy = DynamicTopologyModule(128, 16, 128).to(DEVICE)

_frob_m, _frob_s = topology_init_sensitivity(
    backbone    = _bb_dummy,
    edge_module = _eim_dummy,
    dyn_module  = _dtm_dummy,
    episode_fn  = _dummy_episode_fn,   # Section 13 dummy function
    n_episodes  = 5,
    n_inits     = 3,
    knn_k       = 5,
    embed_dim   = 128,
    rank        = 16,
)
print(f'Untrained D2 sanity: frob_mean={_frob_m:.4f}  frob_std={_frob_s:.4f}')
print('(High frob expected for untrained model -- meaningful only post-training)')
print()
print('To run full Group D: set RUN_GROUP_D=True in Section 15b.')


## Section 15e: Ablation Group B -- Selection Strategy

**Group B** (plan 5.2) tests how the prototype aggregation strategy
affects accuracy, following the FSAKE observation that max-selection
works in 1-shot while nearest-to-centroid works better in 5-shot.

| Variant | Topology | Strategy | 1-shot Acc |
| ------- | -------- | -------- | ---------- |
| B1_max_k5 | Fixed k=5 | Max (mean pool) | (report) |
| B2_nearest_k5 | Fixed k=5 | Nearest-to-centroid | (report) |
| B3_dynamic | Dynamic learned | Auto-adaptive | (report) |

**What this proves**: If B3 >= B1 and B2, dynamic topology subsumes
the need to manually tune selection strategy.

### Implementation
`selection_strategy` parameter added to `DEKAEModel.__init__`.
When `nearest_centroid`, prototype step soft-weights each support node
by `softmax(cosine_sim(node, rough_centroid) * T)` with temperature T=5.


In [ ]:
# -- Section 15e: Group B -- Selection Strategy sanity checks ----------------

_s_b = torch.randn(5, 3, 84, 84).to(DEVICE)
_l_b = torch.arange(5).to(DEVICE)
_q_b = torch.randn(15, 3, 84, 84).to(DEVICE)

print('Group B sanity checks (untrained models):')
print('-' * 60)

for _bname, _strat in [('B1_max_k5', 'max'), ('B2_nearest_k5', 'nearest_centroid')]:
    _mb = build_model(ABLATION_CONFIGS[_bname])
    _mb.eval()
    with torch.no_grad():
        _, _, _met_b = _mb(_s_b, _l_b, _q_b)
    print(f'  {_bname:<22} strategy={_strat:<20} density={_met_b["graph_density"]:.3f}')

# 5-shot: nearest_centroid meaningful only when k_shot > 1
_s_b5 = torch.randn(25, 3, 84, 84).to(DEVICE)  # 5-way 5-shot = 25 support
_l_b5 = torch.repeat_interleave(torch.arange(5).to(DEVICE), 5)
_q_b5 = torch.randn(15, 3, 84, 84).to(DEVICE)

for _strat5 in ('max', 'nearest_centroid'):
    _mb5 = DEKAEModel(n_way=5, embed_dim=128, use_dynamic=False,
                      use_lmt=False, selection_strategy=_strat5).to(DEVICE)
    _mb5.eval()
    with torch.no_grad():
        _lgt_b5, _, _ = _mb5(_s_b5, _l_b5, _q_b5)
    print(f'  5-shot {_strat5:<20}: logits={_lgt_b5.shape} OK')

print()
print('Both strategies run correctly.')
print('To run full Group B training: set RUN_GROUP_B=True in Section 15b.')


## Section 15f: Ablation Groups G1-G5 -- Sensitivity Analysis

**Group G** (plan 5.7) validates robustness across operating conditions.

| Group | What varies | Runner |
| ----- | ----------- | ------ |
| G1 | Noise sigma (synthetic Sec.14) | `run_group_g1_noise()` below |
| G2 | n-way/k-shot configs | `RUN_GROUP_G2=True` in Sec.15b |
| G3 | Backbone embed_dim | `RUN_GROUP_G3=True` in Sec.15b |
| G4 | Dataset / resolution | re-run Sec.13b with each dataset |
| G5 | lambda_edge value | `RUN_GROUP_G5=True` below |

### G4 target table (plan 5.7 G4)
| Dataset | Resolution | HGNN 5-shot | FSAKE 5-shot | Ours 5-shot |
| ------- | ---------- | ----------- | ------------ | ----------- |
| miniImageNet | 84x84 | (report) | 79.66 | (report) |
| CIFAR-FS | 32x32 | 86.16 | 85.92 | (report) |


In [ ]:
# -- Section 15f: Groups G -- Sensitivity Analysis ---------------------------


# ---------------------------------------------------------------------------
# G1: Noise-level robustness sweep (sigma in Section 14 synthetic experiment)
# ---------------------------------------------------------------------------
def run_group_g1_noise(
    sigmas=(0.3, 0.6, 0.8, 1.0),
    n_trials=20,
    n_nodes_per_class=5,
    n_classes=5,
    embed_dim=128,
):
    """Sweep sigma and run Section-14 synthetic graph-recovery experiment.

    Delegates to ``synthetic_recovery_experiment`` (Section 14) which already
    computes both the static k-NN baseline and the DEKAE dynamic topology in
    a single pass over each sigma value.

    Returns a list of result dicts (one per sigma value).
    """
    df = synthetic_recovery_experiment(
        n_way=n_classes,
        n_per_class=n_nodes_per_class,
        feat_dim=embed_dim,
        sigma_values=tuple(sigmas),
        n_trials=n_trials,
    )
    if df is None:
        print('G1 skipped -- pandas not available.')
        return []

    results = []
    print('Group G1 -- Noise Robustness Sweep')
    print('=' * 65)
    print(f'  {"Sigma":<8}  {"Static F1":>10}  {"DEKAE F1":>10}')
    print('-' * 65)

    for _, row in df.iterrows():
        entry = {
            'sigma'      : row['sigma'],
            'static_f1'  : row['kNN_F1_mean'],
            'dynamic_f1' : row['DEKAE_F1_mean'],
            'static_f1_std' : row['kNN_F1_std'],
            'dynamic_f1_std': row['DEKAE_F1_std'],
        }
        results.append(entry)
        print(f'  {entry["sigma"]:<8.2f}  {entry["static_f1"]:>10.3f}  '
              f'{entry["dynamic_f1"]:>10.3f}')

    print('=' * 65)
    import json as _jg1
    _out_g1 = str(RESULTS_DIR / 'ablation_group_g1.json')
    with open(_out_g1, 'w') as _fg1:
        _jg1.dump(results, _fg1, indent=2)
    print(f'G1 results saved to {_out_g1}')
    return results


# ---------------------------------------------------------------------------
# G5: lambda_edge sensitivity
# ---------------------------------------------------------------------------
RUN_GROUP_G5 = False   # set True to sweep lambda_edge
if RUN_GROUP_G5:
    print('\nRunning Group G5 ablations (lambda_edge sensitivity)...')
    group_g5_results = run_all_ablations(
        groups           = ['G5_lam0', 'G5_lam01', 'G5_lam05', 'G5_lam1', 'G5_lam2'],
        episode_fn_train = train_sampler_aug,
        episode_fn_val   = val_sampler,
    )
    print_ablation_table(group_g5_results, 'Group G5 -- Lambda_edge Sensitivity')
    import json as _jg5
    _out_g5 = str(RESULTS_DIR / 'ablation_group_g5.json')
    with open(_out_g5, 'w') as _fg5:
        _jg5.dump([{k: v for k, v in r.items() if k != 'per_episode'}
                   for r in group_g5_results], _fg5, indent=2)
    print(f'G5 results saved to {_out_g5}')
else:
    print('  (Group G5 skipped -- set RUN_GROUP_G5=True)')


# Summary
print()
print('Group G functions / runners ready:')
print('  G1 run_group_g1_noise() -- sigma sweep on synthetic experiment')
print('  G2 RUN_GROUP_G2=True    -- n-way/k-shot in ABLATION_CONFIGS')
print('  G3 RUN_GROUP_G3=True    -- embed_dim in ABLATION_CONFIGS')
print('  G4 documented above     -- re-run Section 13b per dataset')
print('  G5 RUN_GROUP_G5=True    -- lambda_edge sensitivity (this cell)')
print()
print('Group B:')
print('  selection_strategy in DEKAEModel; B1/B2/B3 in ABLATION_CONFIGS')
print('  Section 15e sanity check passed; RUN_GROUP_B=True in Sec.15b')


## Section 16: Metrics Logging (Weights & Biases)

W&B is the recommended logger to survive Colab session disconnects. A JSON backup is saved to Drive at the end of every epoch as a fallback.

In [ ]:
# ── Section 16: Metrics Logging (W&B) ────────────────────────────────────────
# Run this cell to initialise W&B before calling train().
# Set LOG_WANDB=True in train() call to enable cloud sync.

try:
    import wandb

    WANDB_PROJECT = "DEKAE-FSL"
    WANDB_ENTITY  = None          # ← set to your W&B username/team if needed

    def init_wandb(run_name: str = "dekae_run", cfg: dict = None):
        wandb.init(
            project = WANDB_PROJECT,
            entity  = WANDB_ENTITY,
            name    = run_name,
            config  = cfg or CFG,
            resume  = "allow",    # resume if run_id matches previous session
        )
        print(f"W&B run initialised: {wandb.run.url}")

    # ── Full metrics schema (plan §6.2) ───────────────────────────────────────
    # Matches the log dict in train() exactly — edit both together if adding keys.
    METRIC_SCHEMA = {
        "train_loss"          : None,   "train_acc"           : None,
        "val_acc"             : None,   "test_acc"            : None,
        "avg_degree"          : None,   "degree_std"          : None,   # plan §6.1 & §6.2
        "edge_entropy"        : None,   "graph_density"       : None,   # watch for collapse
        "topology_stability"  : None,   "intra_edge_ratio"    : None,   # plan §6.1 (density std)
        "inter_edge_ratio"    : None,   "layer_stability"     : None,   # layer_repr_stability
        "grad_norm_mean"      : None,   "lambda_edge_scale"   : None,
        "num_parameters"      : None,   "epoch_time_s"        : None,   # plan §6.2
    }

    print("W&B available. Call init_wandb('run_name') before train().")
    print("Metric keys:", list(METRIC_SCHEMA.keys()))

except ImportError:
    print("W&B not installed. Fallback: JSON logs saved to:", RESULTS_DIR)

# ── Session-disconnect guard: flush & save ────────────────────────────────────
def safe_flush(history: list, run_name: str):
    """Save history to JSON and flush W&B (if active). Call from training loop."""
    path = RESULTS_DIR / f"{run_name}_history.json"
    with open(str(path), "w") as f:
        json.dump(history, f, default=str)
    try:
        import wandb
        if wandb.run is not None:
            wandb.save(str(path))
    except Exception:
        pass


print("safe_flush() registered.")


## Section 17: Visualization Suite

Five required plots: (1) topology per GNN layer, (2) degree distribution, (3) graph density curve, (4) t-SNE before/after, (5) degree vs. sample difficulty using margin difficulty $d_i = s_1(i) - s_2(i)$.

In [ ]:
# ── Section 17: Visualization Suite ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from sklearn.manifold import TSNE

sns.set_theme(style="whitegrid", font_scale=1.1)
SAVE_DIR = RESULTS_DIR / "figures"
SAVE_DIR.mkdir(exist_ok=True)


# ── Plot 1: Graph Topology per GNN Layer ─────────────────────────────────────

def plot_topology(A: torch.Tensor, labels: torch.Tensor, title: str = "",
                  save_path=None):
    """Visualise an episode graph coloured by class label."""
    N = A.size(0)
    A_np = (A > 0.01).float().cpu().numpy()
    G    = nx.from_numpy_array(A_np)

    label_list = labels.cpu().tolist()
    cmap = plt.colormaps.get_cmap("tab10")
    colors = [cmap(l / max(max(label_list), 1)) for l in label_list]

    fig, ax = plt.subplots(figsize=(6, 5))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx(G, pos, node_color=colors,
                     with_labels=False, node_size=120,
                     edge_color="#aaa", width=0.6, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 2: Degree Distribution Histogram ─────────────────────────────────────

def plot_degree_distribution(A: torch.Tensor, title: str = "", save_path=None):
    A_no_diag = A.clone(); A_no_diag.fill_diagonal_(0)
    degrees = (A_no_diag > 0).float().sum(dim=1).cpu().numpy()
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(degrees, bins=range(int(degrees.max()) + 2),
            color="steelblue", edgecolor="white")
    ax.set_xlabel("Node Degree")
    ax.set_ylabel("Count")
    ax.set_title(title or "Degree Distribution")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 3: Graph Density Curve over Training ─────────────────────────────────

def plot_density_curve(history: list, save_path=None):
    epochs    = [h["epoch"] for h in history]
    densities = [h.get("graph_density", 0) for h in history]
    fig, ax   = plt.subplots(figsize=(6, 3))
    ax.plot(epochs, densities, color="coral", linewidth=2)
    ax.axhline(0.5, ls="--", color="red", alpha=0.6, label="Collapse threshold (0.5)")
    ax.axhspan(0.10, 0.15, alpha=0.08, color="green", label="Target density range (0.10–0.15)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Graph Density")
    ax.set_title("Graph Density over Training (monitor for collapse)")
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 3b: Intra/Inter Edge Ratio over Training (NEW) ──────────────────────

def plot_edge_ratio_curve(history: list, save_path=None):
    """
    Plot intra-class vs inter-class edge ratios over training.
    Expected: intra_edge_ratio should INCREASE over training as the model
    learns to preferentially connect same-class nodes via the edge correction
    loss. Flat or decreasing intra ratio = edge loss not engaging.
    """
    epochs = [h["epoch"] for h in history if "intra_edge_ratio" in h]
    if not epochs:
        print("No intra_edge_ratio data in history — re-run training with updated code.")
        return
    intra  = [h["intra_edge_ratio"] for h in history if "intra_edge_ratio" in h]
    inter  = [h["inter_edge_ratio"] for h in history if "inter_edge_ratio" in h]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(epochs, intra, color="#2ecc71", linewidth=2, label="Intra-class edge ratio")
    ax.plot(epochs, inter, color="#e74c3c", linewidth=2, label="Inter-class edge ratio",
            linestyle="--")
    ax.axhline(1.0 / 5, ls=":", color="gray", alpha=0.7,
               label="Random baseline (1/n_way = 0.20)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Edge Ratio (support-support edges)")
    ax.set_title("Intra/Inter Edge Ratio over Training\n"
                 "(Rising intra ratio = edge loss building class-consistent topology)")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 4: t-SNE Before / After Dynamic Topology ────────────────────────────

def plot_tsne(H_before: torch.Tensor, H_after: torch.Tensor,
              labels: torch.Tensor, save_path=None):
    H_all  = torch.cat([H_before, H_after], dim=0).cpu().detach().numpy()
    labels_np = labels.cpu().numpy()
    tsne   = TSNE(n_components=2, perplexity=15, random_state=42)
    emb    = tsne.fit_transform(H_all)
    N      = H_before.size(0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, title, start, end in zip(
            axes,
            ["Before (H⁰)", "After Dynamic Topology (Hᴸ)"],
            [0, N], [N, 2 * N]):
        sc = ax.scatter(emb[start:end, 0], emb[start:end, 1],
                        c=labels_np, cmap="tab10", s=40, alpha=0.8)
        ax.set_title(title); ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 5: Node Degree vs. Sample Difficulty ─────────────────────────────────

def plot_degree_vs_difficulty(model: DEKAEModel, episode_fn,
                               n_way: int, k_shot: int, n_query: int,
                               n_episodes: int = 600, save_path=None):
    """
    Bin query nodes into easy / medium / hard tertiles by margin difficulty
    d_i = s1(i) - s2(i)  (smaller margin = harder sample).
    Show median node degree per bin with 95% CI error bars.
    """
    model.eval()
    margin_diffs, degrees = [], []

    with torch.no_grad():
        for _ in tqdm(range(n_episodes), desc="Difficulty analysis"):
            s_imgs, s_lbl, q_imgs, q_lbl = episode_fn(n_way, k_shot, n_query)
            s_imgs = s_imgs.to(DEVICE); s_lbl = s_lbl.to(DEVICE)
            q_imgs = q_imgs.to(DEVICE)

            logits, _, mets = model(s_imgs, s_lbl, q_imgs)

            sorted_logits, _ = logits.sort(dim=1, descending=True)
            margin = (sorted_logits[:, 0] - sorted_logits[:, 1]).cpu().numpy()
            margin_diffs.extend(margin.tolist())

            avg_d = mets.get("avg_degree", 0)
            degrees.extend([avg_d] * logits.size(0))

    margin_arr  = np.array(margin_diffs)
    degree_arr  = np.array(degrees)

    tertile1, tertile2 = np.percentile(margin_arr, [33, 66])
    bins = {"Hard\n(low margin)"  : degree_arr[margin_arr <= tertile1],
            "Medium"              : degree_arr[(margin_arr > tertile1) & (margin_arr <= tertile2)],
            "Easy\n(high margin)" : degree_arr[margin_arr > tertile2]}

    labels_plot  = list(bins.keys())
    means  = [np.median(v) for v in bins.values()]
    cis    = [1.96 * np.std(v) / max(np.sqrt(len(v)), 1) for v in bins.values()]

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(labels_plot, means, yerr=cis, color=["#e74c3c", "#f39c12", "#2ecc71"],
           capsize=6, edgecolor="white")
    ax.set_ylabel("Average Node Degree")
    ax.set_title("Node Degree vs. Sample Difficulty\n(Hard samples → more neighbours)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


# ── Plot 6: Adjacency Heatmap — Static k-NN vs. Learned Dynamic ──────────────

def plot_adjacency_heatmap(A_dynamic: torch.Tensor, A_static: torch.Tensor,
                            labels: torch.Tensor, save_path=None):
    """
    Side-by-side heatmap comparing the static k-NN adjacency to the
    learned dynamic adjacency for the same episode.

    Nodes are sorted by class label so intra-class block structure becomes
    visually apparent.  A well-trained DEKAE model should show stronger
    block-diagonal structure in A_dynamic vs. A_static.

    Plan reference: §6.5 visualisation item 6.

    Parameters
    ----------
    A_dynamic : (N, N) float tensor — learned dynamic adjacency A'(L)
    A_static  : (N, N) float tensor — static k-NN adjacency A_init
    labels    : (N,) long tensor    — node class indices (for reordering)
    save_path : optional file path to save the figure
    """
    try:
        import seaborn as sns
    except ImportError:
        print("seaborn not available — install with: pip install seaborn")
        return

    N = A_dynamic.size(0)
    sort_idx   = torch.argsort(labels).cpu().numpy()
    A_dyn_np   = A_dynamic.cpu().detach().numpy()[sort_idx][:, sort_idx]
    A_stat_np  = A_static.cpu().detach().numpy()[sort_idx][:, sort_idx]
    labels_np  = labels.cpu().numpy()[sort_idx]

    # Build tick positions at class boundaries
    unique, counts = np.unique(labels_np, return_counts=True)
    boundaries = np.cumsum(counts)[:-1] - 0.5

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, mat, title in zip(
            axes,
            [A_stat_np, A_dyn_np],
            ["Static k-NN Adjacency A_init", "Learned Dynamic Adjacency A'(L)"]):
        sns.heatmap(mat, ax=ax, cmap="Blues", vmin=0, vmax=1,
                    xticklabels=False, yticklabels=False,
                    cbar_kws={"shrink": 0.8})
        # Overlay class boundary lines
        for b in boundaries:
            ax.axhline(y=b, color="red", linewidth=0.8, linestyle="--", alpha=0.7)
            ax.axvline(x=b, color="red", linewidth=0.8, linestyle="--", alpha=0.7)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel(f"N={N} nodes (sorted by class)")

    fig.suptitle("Adjacency Comparison: Static vs. Dynamic\n"
                 "(Block-diagonal = class-consistent topology; red lines = class boundaries)",
                 fontsize=11)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()


# ── Demo: run all plots on the toy model ─────────────────────────────────────

# Plot 1 — episode topology (coloured by class)
_labels_demo = torch.cat([torch.full((4,), c) for c in range(5)]).to(DEVICE)
plot_topology(_A_dyn, _labels_demo, title="Episode Topology (Layer 3)",
              save_path=str(SAVE_DIR / "topology_layer3.png"))

# Plot 2 — degree distribution
plot_degree_distribution(_A_dyn, "Degree Distribution (DEKAE)",
                         save_path=str(SAVE_DIR / "degree_dist.png"))

# Plots 3/3b/4/5 require training history & real data — run after train()
print("\nPlots 3/3b/4/5/6 require training history & real data — run after train().")
print("After training, call:")
print("  plot_density_curve(train_history)")
print("  plot_edge_ratio_curve(train_history)   ← NEW: monitors edge-loss effectiveness")
print("  plot_tsne(H_before, H_after, labels)")
print("  plot_degree_vs_difficulty(eval_model, test_sampler, n_way=5, k_shot=1, n_query=15)")
print("  plot_adjacency_heatmap(A_dyn, A_static, labels) ← plan §6.5 item 6")
